# <center>Agent 智能体部署上线</center>

&emsp;&emsp;前面几节课我们把 ai-todo 这套 FastAPI + LangGraph 应用在本机调通了——开发模式下 `localhost:12234` 一打开，前端、聊天、SSE 都跑得很顺。但这一刻它还困在自己的笔记本电脑里：朋友打不开、手机访问不了，所有演示一关机就没了。本门课要做的事情就是：把 ai-todo 从本机真正搬到 `https://myagent-lab.online` 这个公网域名上，让任何人打开浏览器都能用上我们写的智能体。

&emsp;&emsp;本机到公网之间隔着 VPS、域名、HTTPS 这一整套基础设施。下面六章会按顺序把每个环节逐一接通——从 Ubuntu 基线到代码上线、systemd 守护、HTTPS 加密、Nginx反向代理、日志监控,最后串成一条可复用的部署路径。

<div align=center><font size=2 color=#999999>课程封面：从本机到 https://域名 的完整搬迁</font></div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/agent-deploy/2026-05-15/opening-deploy-overview-b3093fab.png" width=80% alt="课程封面：从本机到 https://域名 的完整搬迁"></div>

<br>

&emsp;&emsp;**第一章** 把 Ubuntu 24.04 基线装包与配置做对；**第二章** 把代码搬上服务器、配 `.env` 安全、用 uvicorn 在公网先裸跑一遍；**第三章** 交给 systemd 守护进程；**第四章** 用 nginx 做反代入口、把 HTTPS 装上；**第五章** 把日志轮转、健康检查、监控入口三件事一次配齐；**第六章** 回顾全部产物 + 整体架构图 + 速查。每一章末尾都会留下一个可直接复用的产物，最终六件产物拼成一条完整的部署路径。

&emsp;&emsp;走完这 6 章我们会带走 6 件可直接复用的产物——HTTPS 已上的 ai-todo 公网服务、systemd 守护的进程、nginx 完整 site config、certbot 自动续期、logrotate 防爆盘、`/health` 探针入口。前置依赖是已完成"接口设计篇"那一节课程的学习（能用 FastAPI 写 HTTP 接口、对 SSE 与 LangGraph 的工作机制有印象、知道 API Key 怎么走鉴权 header），技术上需要一台已购的腾讯云轻量香港 VPS 加一个国际域名（本门课用 `.online`，走港区免备案直连路径）——VPS 购买和域名注册的图文步骤独立在 `课前准备.md`，请自学完成后再进第一章。全部命令与配置截止 2026 年 5 月初，基于 Ubuntu 24.04 LTS + nginx 1.24 + systemd 255 + certbot 5.x snap 渠道，本科上线部署在腾讯云香港 VPS（`43.129.193.131`）上跑通。

---

## <center>第一章 Ubuntu 24.04 基线装包与配置</center>

&emsp;&emsp;课前准备里我们已经买好了腾讯云轻量香港区 + `.online` 国际域名——走的是港区免备案直连路径。`.online` 是国际域不用 <font color=red>ICP 备案</font>，机房在香港也不在大陆工信部管辖范围内。对比大陆机房 + 中国域名的"正规"路径，大陆路径在拿到域名后必须先做 ICP 备案才能对外解析，通常 2-3 周——这段等待太长，所以我们锁定港区路径，把上线节奏抓在自己手里。如果将来要承接大陆 to-C 业务，备案是绕不过去的合规要求。

> <font size=2>【名词解释】<b><font color=red>ICP 备案</font>(Internet Content Provider Filing，互联网信息服务提供者备案)</b>:中国大陆对境内互联网服务做实名登记的合规制度,境内机房必须先备案才能解析域名。本课走港区路径就避开了。</font>

&emsp;&emsp;现在我们手里就一台刚买的腾讯云轻量香港 VPS——干净的 Ubuntu 24.04，什么都没装。第一件事不是急着上传代码，而是先把"基线装包"和"SSH 接入"两件事做对——基线是后面所有 Python 跑起来的地基，SSH 接入则决定了我们登服务器的方式有没有留隐患。

&emsp;&emsp;基线装包是让服务器具备跑 ai-todo 的最低条件，SSH 接入是我们从外面登进去的唯一通道。课堂为了演示节奏顺畅全程走密码登录，正式上线推荐改密钥并禁密码的加固命令我们留到最后一章附录。

<div align=center><font size=2 color=#999999>基线装包与 SSH 接入双线路径</font></div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/agent-deploy/2026-05-15/L2-baseline-ssh-1cf3c6b2.png" width=80% alt="基线装包与 SSH 接入双线路径"></div>

<br>

&emsp;&emsp;基础环境我们要装两类东西：一是跑 ai-todo 所需的 Python 3.12 + git + curl + snapd，二是为后面要用的 certbot 准备好 snap 通道。同时还要把"课堂用密码登录"和"正式上线推荐密钥加固"的区别讲清楚——课堂为了演示方便我们全程用密码，但如果接下来要真的把这台服务器长期对外开放，加固命令在最后一章会以附录形式给到。

### 1.1 SSH 登服务器

&emsp;&emsp;课前准备里我们已经在腾讯云控制台拿到了 VPS 的公网 IP 和系统默认用户 `ubuntu` 的密码。第一次登服务器走 `ssh ubuntu@公网IP` 这条最朴素的路径——下面把命令在 macOS / Linux 和 Windows PowerShell 两条路径上分别列一遍。注意 Windows 10/11 已经内置 OpenSSH 客户端，PowerShell 里直接 `ssh` 命令就能用，跟 Linux 体感一致。

```bash
# macOS / Linux / Git Bash（本机）：第一次 SSH 登 VPS
# 把 43.129.193.131 换成你自己 VPS 的公网 IP
ssh ubuntu@43.129.193.131
# 首次连接会提示 "The authenticity of host ... can't be established"，输 yes 回车
# 然后输入腾讯云控制台拿到的 ubuntu 用户密码（粘贴时屏幕不显示是正常的）
```

```bash
# Windows PowerShell（本机）：第一次 SSH 登 VPS
ssh ubuntu@43.129.193.131
# Windows 10/11 内置 OpenSSH 客户端，命令完全一致
# 首次连接同样会问 yes / no，输 yes 回车，再输密码
```

&emsp;&emsp;登成功后命令行提示符会从本机的 `username@laptop` 变成 `ubuntu@<服务器主机名>`——这表示我们已经在 VPS 内部了，下面所有 `apt`、`sudo` 命令都是在服务器上跑、不是在本机上跑。退出回本机用 `exit` 或 `Ctrl+D`。后续章节我们会在 1.5 配本机 SSH alias 这一节把这条长命令简化成 `ssh myagent`。

### 1.2 装服务器基线包

&emsp;&emsp;SSH 登进 VPS 之后，跑一遍 `apt update && apt upgrade` 让系统包都到最新版本，然后装四个核心包。Python 3.12 是 Ubuntu 24.04 自带的版本，但默认 image 不一定装了 `venv` 模块，所以要显式装 `python3.12-venv`。snapd 是为了后面装 certbot 准备的——certbot 在 Ubuntu 24.04 的官方推荐渠道是 snap，不是 apt。

```bash
# 服务器：基线装包
sudo apt update
sudo apt upgrade -y
sudo apt install -y python3.12 python3.12-venv git curl snapd
python3.12 --version    # 期望输出 Python 3.12.x
```

&emsp;&emsp;这一段命令跑完，服务器就具备了运行 Python 应用的最低基线。验证一句 `python3.12 --version` 出 `Python 3.12.x` 表示 Python 装好了。后面所有 venv 操作、所有 systemd unit 里的 Python 路径，都依赖这一行。

### 1.3 防火墙现状说明

&emsp;&emsp;Ubuntu 24.04 自带主机层 ufw 防火墙，默认状态是 inactive。原因是腾讯云轻量服务器在云控制台已经有一层"安全组/防火墙"，默认放行 22/80/443 三个端口，这一层是物理上拦在云入口的，已经覆盖了我们需要的入口控制。Ubuntu 主机层 ufw 是再加一层冗余，正式工程上上线推荐启用作为纵深防御。

&emsp;&emsp;**第二、三章我们要临时多开两个端口**——第二章 ai-todo 公网裸跑需要 12233(前端临时 web server)和 12234(后端 uvicorn)对公网可达,云控制台默认只放行 22/80/443,所以现在就要去控制台手动加这两条入站规则。腾讯云轻量路径:**控制台 → 防火墙 → 添加规则 → TCP / 12233 / 来源 0.0.0.0/0**,再加一条 12234 的。第四章 nginx 接管入口之后,这两个端口就会收回到 loopback,届时可以把这两条规则删掉,公网入口只剩 22/80/443 三件。

> <font size=2>【名词解释】<b><font color=red>loopback</font>(loopback interface,回环接口)</b> — `127.0.0.1` 对应的虚拟网卡,数据包在内核里就被路由回本机、不出物理网卡。后端只监听 loopback 等于物理隔离,公网完全摸不到,只能从本机的进程(本课是 nginx)进入。</font>

> <font size=2>【名词解释】<b><font color=red>ufw</font>(Uncomplicated Firewall，简易防火墙)</b>:Ubuntu 自带的主机内防火墙命令工具,底层是 iptables。跟云控制台的安全组并列,两者各管一段、互为冗余。</font>

### 1.4 配本机 SSH alias

&emsp;&emsp;最后一件便利性的事——给本机 `~/.ssh/config` 加一段 Host alias，后续 `ssh myagent` 就能直接替代 `ssh -p 22 ubuntu@myagent-lab.online` 这种长命令。这一步纯为了减少手敲长命令时打错的概率。

```bash
# macOS / Linux / Git Bash（本机，不是服务器）：配 SSH alias
cat >> ~/.ssh/config <<'EOF'
Host myagent
    HostName myagent-lab.online
    User ubuntu
EOF
```

&emsp;&emsp;上面 heredoc 复制到 `EOF` 闭合行就停手,别把下面 chmod 一起选中。下面这条单独跑——OpenSSH 客户端硬约束 `~/.ssh/config` 权限过松(组/其他可读)会被直接拒用,后面 1.6 配公钥免密那一步会奇怪地失效,这一行漏了排查会绕很久:

```bash
# chmod 600 = 属主可读写(6=4读+2写)、组无权限(0)、其他用户无权限(0)
chmod 600 ~/.ssh/config
```

```bash
# Windows PowerShell（本机）：配 SSH alias
$sshDir = "$HOME\.ssh"
if (-not (Test-Path $sshDir)) { New-Item -ItemType Directory -Path $sshDir | Out-Null }
$sshConfig = "$sshDir\config"
if (-not (Test-Path $sshConfig)) { New-Item -ItemType File -Path $sshConfig | Out-Null }
@"
Host myagent
    HostName myagent-lab.online
    User ubuntu
"@ | Add-Content -Path $sshConfig -Encoding ASCII
# Windows ACL 等价 chmod 600：去继承 + 只给当前用户完全控制
icacls $sshConfig /inheritance:r /grant:r "$($env:USERNAME):F" | Out-Null
```

&emsp;&emsp;配完之后，本机一句 `ssh myagent` 就能登服务器。Windows 10/11 内置 OpenSSH 客户端，`ssh` / `scp` / `ssh-keygen` 都可以在 PowerShell 里直接用——位置跟 macOS 一致放在 `~/.ssh/`（PS 写法 `$HOME\.ssh\`）。后续课程里所有"本机 → 服务器"的 SSH/scp 命令都以这个 alias 为前提。

### 1.5 配公钥免密

&emsp;&emsp;1.4 配本机 SSH alias 这一节配好的 `ssh myagent` 还会要密码,因为 OpenSSH 故意不允许把密码写进配置文件(被读到等于密码泄露)。本节用**公钥认证**替代密码:本机生成一对密钥,公钥贴到服务器 `~/.ssh/authorized_keys`,登录时服务器用公钥验证私钥签名,对得上就放行。

&emsp;&emsp;**本节只加公钥、不关密码登录**——保留密码作为逃生通道(私钥丢了或权限设坏了至少还能进)。如果以后要把入口完全收紧(关掉密码登录和 root 登录),那是另外的工程加固话题,本课不展开,生产环境上线前再单独加固。

> <font size=2>【名词解释】<b><font color=red>公钥认证</font>(Public Key Authentication，SSH 免密登录的标准做法)</b>:本机生成一对非对称密钥,公钥贴到服务器、私钥留在本机,登录时服务器用公钥验证本机私钥签名,对得上就放行。比密码登更安全(没有可爆破的密码),也省去每次手敲密码。</font>

&emsp;&emsp;开手前先纠正一个常见误解:**一对密钥代表本机这个身份,不是配给一台服务器的**——同一把 `id_ed25519.pub` 可以推给任意多台服务器、GitHub、GitLab,每个目的地只是把这把公钥多记一行 `authorized_keys` 而已。

&emsp;&emsp;所以 `ssh-keygen` 问 `Overwrite (y/n)?` 时**默认打 n**:选 y 覆盖旧私钥,这把旧公钥已经贴过的地方都立刻登不上,要挨个重新分发新公钥。

&emsp;&emsp;算法选 **ed25519**——OpenSSH 6.5+ (2014) 起支持,密钥短、签名快、安全强度等效 RSA-3072,主流云厂商和代码托管全覆盖。下表把其它三种放进来一起对比:

| 算法 | 推荐度 | 命令 | 说明 |
|---|---|---|---|
| **ed25519** | ⭐ 现代首选 | `ssh-keygen -t ed25519` | 短(~400B)、快、强(等效 RSA-3072)。本课默认 |
| **rsa** | ✅ 兼容兜底 | `ssh-keygen -t rsa -b 4096` | 必须 `-b 4096`。古董 Linux / 嵌入式设备才用 |
| ecdsa | ⚠️ 不推荐 | — | NIST 曲线参数透明度有疑虑(Snowden 之后),没人爱用 |
| dsa | ❌ 已废弃 | — | OpenSSH 7.0 (2015) 起默认禁用 |

> <font size=2>【名词解释】<b><font color=red>ed25519</font>(Edwards-curve Digital Signature Algorithm using Curve25519，基于 Curve25519 椭圆曲线的数字签名算法)</b>:现代 SSH/Git 默认推荐密钥算法,密钥短(私钥 ~400B)、签名快、安全强度等效 RSA-3072,OpenSSH 6.5+ (2014) 起支持。</font>

```bash
# macOS / Linux / Git Bash（本机，不是服务器）：生成密钥对并推到服务器
# ─────────────────────────────────────────────────
# 第一步：先看本机是不是已经有 ed25519 key
ls ~/.ssh/id_ed25519 ~/.ssh/id_ed25519.pub 2>/dev/null
# 如果两个文件都已存在 → 跳过第二步生成，直接到第三步推公钥
# 如果完全没有 → 走第二步生成

# 第二步：生成 ed25519 密钥对
ssh-keygen -t ed25519 -C "myagent-deploy"
# 交互逐行说明（终端实际会一句句问，按下面回答）：
#   Enter file in which to save the key (~/.ssh/id_ed25519):  ← 直接回车，接受默认路径
#   /Users/xxx/.ssh/id_ed25519 already exists.                ← 只有当旧 key 存在才出现这两行
#   Overwrite (y/n)?                                          ← 打 n 回车！除非确定旧 key 没在别处用
#   Enter passphrase (empty for no passphrase):               ← 直接回车，留空（本地开发够用）
#   Enter same passphrase again:                              ← 直接回车，留空
# ⚠️ "Overwrite y" 会覆盖旧私钥——如果旧 key 已经登过 GitHub / 其他 VPS，那些地方会立刻登不上

# 第三步：把公钥推到服务器（这一次仍要输服务器密码，之后就免密了）
ssh-copy-id myagent

# 第四步：验证免密：应该直接进服务器，不再问密码
ssh myagent
```

> ⚠️ **不小心选了 `Overwrite y` 怎么办**：旧 ed25519 已经被覆盖，所有用旧公钥的地方都登不上了。修法是把新公钥重新分发：
> - **GitHub**：`pbcopy < ~/.ssh/id_ed25519.pub`(Linux 用 `xclip -sel c`)然后到 https://github.com/settings/keys 删旧加新
> - **GitLab / Gitee**：同上路径，对应 Settings → SSH Keys
> - **其他 VPS**：用密码登上去，把 `~/.ssh/authorized_keys` 里的旧公钥行替换成新的
> - **不确定旧 key 在哪用过**：先放着，遇到 `Permission denied (publickey)` 再处理就行

```bash
# Windows PowerShell（本机，不是服务器）：生成密钥对并推到服务器
# ─────────────────────────────────────────────────
# 第一步：先看本机是不是已经有 ed25519 key
Test-Path "$HOME\.ssh\id_ed25519", "$HOME\.ssh\id_ed25519.pub"
# 两个 True → 跳过第二步生成，直接到第三步推公钥
# 有 False → 走第二步生成

# 第二步：生成 ed25519 密钥对
ssh-keygen -t ed25519 -C "myagent-deploy"
# 交互逐行说明（同 macOS）：
#   Enter file in which to save the key ($HOME\.ssh\id_ed25519):  ← 回车接受默认
#   $HOME\.ssh\id_ed25519 already exists.                          ← 仅旧 key 存在才出现
#   Overwrite (y/n)?                                                ← 打 n！除非确定旧 key 没在别处用
#   Enter passphrase (empty for no passphrase):                     ← 回车留空
#   Enter same passphrase again:                                    ← 回车留空

# 第三步：Windows 没内置 ssh-copy-id，用 Get-Content + ssh 等价完成（这次仍要输服务器密码）
Get-Content "$HOME\.ssh\id_ed25519.pub" | ssh myagent "mkdir -p ~/.ssh && cat >> ~/.ssh/authorized_keys && chmod 600 ~/.ssh/authorized_keys"

# 第四步：验证免密：应该直接进服务器
ssh myagent
```

&emsp;&emsp;**已有 RSA key (`~/.ssh/id_rsa`) 想复用**:`ssh-copy-id myagent` 默认会把它能找到的所有公钥都推到 `authorized_keys`,多一行不冲突,不必为这门课额外生成 ed25519。

&emsp;&emsp;现在 `ssh myagent` 直接进,后面所有 SSH/scp 命令都不再问密码。密码登录依然开着——这只是给入口加了第二条放行路径,不是关掉原路径。

&emsp;&emsp;基线装好了，SSH 入口也清晰了——既有密码登可用，本机也已经通过公钥免密。**从"裸 VPS、什么都没装"到"Python 3.12 + git + curl + snapd 基线 + SSH alias + 公钥免密都配好"——这一章我们让服务器具备了跑 Python 应用的最低条件。** 下一章我们把 ai-todo 代码搬到服务器，先用最朴素的方式让公网能直接打开 HTTP——这样我们就能眼见为实地看到"HTTP 不安全到底不安全在哪"，为后面 nginx 大章里的 HTTPS 推导做铺垫。

---

## <center>第二章 代码上传 + .env 安全 + uvicorn 公网裸跑</center>

&emsp;&emsp;基线装好之后,我们要把本机调好的 ai-todo 真正搬上去——但搬之前有两件事要分开处理:`.env` 里的 <font color=red>OPENROUTER_API_KEY</font> 是访问 LLM 网关的凭证,跟代码 tarball 分开走单独的 scp 通道、单独 `chmod 600` 限权,避免泄露被刷额度;uvicorn 这一节先在 `0.0.0.0:12234` 上跑一遍做最小验证(前端用 `python -m http.server` 临时跑在 12233),等到第四章 nginx 配好再把这两个端口都收回到 loopback。先让 HTTP 通,我们才能亲眼看到"HTTP 不安全到底不安全在哪",为后面 HTTPS 推导做铺垫。

&emsp;&emsp;代码推上去、`.env` 锁紧、uvicorn 暂时监听 `0.0.0.0:12234`、前端用 `python -m http.server` 临时跑在 `0.0.0.0:12233`——四件事配齐之后,浏览器打开 `http://公网IP:12233` 就能看到 ai-todo 前端,前端 JS 跨域调 `公网IP:12234` 的后端(CORS 中间件放行)。这是本课的第一个可观测里程碑。

<div align=center><font size=2 color=#999999>本地代码经 scp 上传到服务器后 uvicorn 公网监听</font></div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/agent-deploy/2026-05-15/L3-upload-uvicorn-a7654476.png" width=80% alt="本地代码经 scp 上传到服务器后 uvicorn 公网监听"></div>

<br>

&emsp;&emsp;这一章我们把 ai-todo 真正跑到服务器上。目标是浏览器直接打开 `http://公网IP:12233` 能看到 ai-todo 前端,前端 JS 跨域调 `公网IP:12234` 的 `/health` 返回 200——也就是 HTTP 通了。本章的"前端 12233 + 后端 12234 双进程双端口"是<font color=red>中间状态</font>:先让 HTTP 跑通,后面 nginx 上线再统一收口。一旦 HTTP 通了,我们就能亲眼看到"HTTP 直连不安全"——这是第四章 HTTPS 推导的起点;HTTPS 配好之后,我们回到这台服务器把后端的 host 从 `0.0.0.0` 收回 `127.0.0.1`(端口 12234 不变,届时后端已被第三章升级到 gunicorn + uvicorn worker)、kill 临时前端 web server、由 nginx 统一在 80/443 接管入口,12233 和 12234 公网入口都退役。

&emsp;&emsp;本章我们要走过 4 个步骤：打包传代码、装依赖、安置 `.env`、公网裸跑验证。ai-todo 自带的 `/health` 端点是部署层健康检查入口，正常返 200、有问题返 503 + `issues` 列表(env 缺失 / 数据库连不上等)——它在第四章 nginx 反代和第五章监控接入时都会用到，本章先把它当作"裸跑通了没"的快速验证手段。

### 2.1 scp 上传代码

&emsp;&emsp;本机用 `tar` 打包项目，排除 `.git`、`.venv`、`.env`、`__pycache__`、`*.pyc` 这五类目录或文件。`.git` 排除是为了不把本地 git history 带上去；`.venv` 排除是因为本地 venv 是 macOS 或 Linux 各自编译的二进制，服务器要重新建；`.env` 单独排除是因为它要走单独的 scp 通道、单独 `chmod 600`，跟代码 tarball 一起传容易丢失这层意图。

```bash
# macOS / Linux / Git Bash（本机）：打包并排除敏感目录
cd ~/projects/ai-todo
tar --exclude='.git' --exclude='.venv' --exclude='__pycache__' \
    --exclude='*.pyc' --exclude='.env' \
    -czf /tmp/ai-todo.tar.gz .

# 把 tarball 和 .env 分两条命令传，便于排错
scp /tmp/ai-todo.tar.gz myagent:/tmp/
scp .env myagent:/tmp/ai-todo.env
```

```powershell
# Windows PowerShell(本机):打包并排除敏感目录
# 前置:Win10 1803+ 内置 tar.exe(bsdtar),Win10 1809+ 内置 scp.exe,命令格式跟 GNU tar 兼容
# 示例假设本机项目放在 D:\project\ai-todo;你的路径不同就替换成自己的
cd D:\project\ai-todo
tar.exe --exclude='.git' --exclude='.venv' --exclude='__pycache__' `
    --exclude='*.pyc' --exclude='.env' `
    -czf D:\project\ai-todo.tar.gz .

# 分两条命令传
scp.exe D:\project\ai-todo.tar.gz myagent:/tmp/
scp.exe .env myagent:/tmp/ai-todo.env
```

&emsp;&emsp;两条 scp 命令分别传 tarball 和 .env，把代码和敏感凭证分开走。.env 一会儿要 `chmod 600` 单独限权，跟代码混在一起会丢失这层意图。在 Windows 上如果觉得 PowerShell 反引号续行别扭，推荐直接用 Git Bash 跑 macOS 版命令，行为完全一致。

### 2.2 venv 与装依赖

&emsp;&emsp;服务器端解压 tarball 到 `/home/ubuntu/ai-todo`，然后用 `python3.12 -m venv .venv` 在项目内建一个虚拟环境，所有依赖装在这个 venv 里——这样不会污染系统 Python，也不会跟服务器上别的 Python 项目冲突。

> <font size=2>【名词解释】<b><font color=red>venv</font>(Python Virtual Environment，Python 虚拟环境)</b>:Python 内置的项目隔离机制,在工程目录下建一个独立的依赖目录(默认 `.venv/`)。激活后 pip 只装到这里,不污染系统 Python。</font>

```bash
# 服务器：解压 + venv + 装依赖
mkdir -p /home/ubuntu/ai-todo
cd /home/ubuntu/ai-todo
tar -xzf /tmp/ai-todo.tar.gz
python3.12 -m venv .venv
.venv/bin/pip install --upgrade pip
.venv/bin/pip install -r requirements.txt

# 验证 venv 内 uvicorn 真实存在
ls -l .venv/bin/uvicorn          # 期望：可执行文件存在
.venv/bin/uvicorn --version      # 期望：打印版本号
```

&emsp;&emsp;这一段跑完，ai-todo 的依赖就全部装在 `/home/ubuntu/ai-todo/.venv/` 里了。后面 systemd 启动应用时要走绝对路径 `/home/ubuntu/ai-todo/.venv/bin/uvicorn` 或 `gunicorn`，依赖的就是这个 venv。

### 2.3 .env 安全管理

&emsp;&emsp;.env 里有 `OPENROUTER_API_KEY` 这种敏感凭证(OpenRouter 网关上调 LLM 用,泄露后别人能刷你的 token 配额),处理它的纪律有四件：① <font color=red>绝不 `git push`</font>（.gitignore 必须包含 `.env`）；② scp 单独传（SSH 端到端加密，中间路由器看不到内容）；③ 服务器上 `chmod 600` 收紧权限（只允许属主读写）；④ 本机项目装 `git-secrets` pre-commit hook 防止误提交。

```bash
# 服务器：安置 .env
mv /tmp/ai-todo.env /home/ubuntu/ai-todo/.env
# chmod 600 = 属主可读写(6=4读+2写)、组无权限(0)、其他用户无权限(0)
# 含义：只有当前用户 ubuntu 能读写这个文件，同机器上的别的账号 cat 不到 OPENROUTER_API_KEY
chmod 600 /home/ubuntu/ai-todo/.env
ls -l /home/ubuntu/ai-todo/.env    # 期望 -rw------- ubuntu ubuntu
```

&emsp;&emsp;为什么 scp 这一步在公开 wifi 也能安全？SSH 协议在客户端到服务器之间建立端到端加密通道，scp 走的是 SSH 隧道，所以中间路由器、wifi 接入点、ISP 看到的都是密文。这跟 HTTP 明文传输是两个不同的安全模型——下一章 nginx 大章我们会专门聊为什么 HTTP 不安全，到时候可以回头对比 SSH 的加密机制和 HTTPS 是同一种思路。

### 2.4 uvicorn 公网裸跑验证

&emsp;&emsp;前面四步都做完,我们要让 ai-todo 真正在公网上响应——分两个进程:后端 uvicorn 监听 `0.0.0.0:12234`、前端用 `python -m http.server` 临时监听 `0.0.0.0:12233`。这个组合跟本机 dev 模式(前端 12233 + 后端 12234)的端口分配完全一致,只是搬到了服务器。这是本章的中间状态,第四章 nginx 接手后两个端口都会收回到 loopback,公网入口只剩 80/443。

```bash
# 服务器终端 A：起后端 uvicorn 监听 0.0.0.0:12234
cd /home/ubuntu/ai-todo
.venv/bin/uvicorn main:app --host 0.0.0.0 --port 12234
```

```bash
# 服务器终端 B：起前端临时 web server 监听 0.0.0.0:12233
cd /home/ubuntu/ai-todo
python3 -m http.server 12233 --directory frontend
```

&emsp;&emsp;两个进程都起好之后，另开一个终端验证。先在服务器上 curl 本机回环、再从本机 curl 公网 IP——后端 12234 返 `{"status":"ok"}` 就说明 FastAPI 在公网真的可达；浏览器打开 `http://43.129.193.131:12233` 就能看到 ai-todo 前端，JS 跨域调 `公网IP:12234`（CORS 中间件已经在 `main.py` 配好放行）。

```bash
# 服务器同一台机器：本机回环验证后端
curl http://127.0.0.1:12234/health                      # 期望 {"status":"ok"}

# macOS / Linux / Git Bash（自己电脑）：公网验证后端
curl http://43.129.193.131:12234/health                 # 期望 {"status":"ok"}
```

```powershell
# Windows PowerShell（自己电脑）：公网验证后端
curl.exe http://43.129.193.131:12234/health             # 期望 {"status":"ok"}
```

&emsp;&emsp;curl 通了之后,**最关键的一步是在自己电脑的浏览器里打开前端**(服务器命令行没浏览器,这一步要在自己电脑做)。地址栏输入 `http://43.129.193.131:12233` 回车,应该看到 ai-todo 的三栏 UI——左侧会话列表、中间聊天框、右侧月历 + 当日待办;输入一句话发出,AI 会流式返回响应,这是前端 JS 跨域 fetch 后端 12234 + CORS 中间件放行 + SSE 透传整条链路第一次端到端跑通。打开浏览器开发者工具(F12)的 Network 标签,能看到所有请求实际打到 `http://43.129.193.131:12234`。

&emsp;&emsp;**顺手再用我们买好的域名打开一次**:地址栏输入 `http://myagent-lab.online:12233` 回车,看到的是一模一样的前端——因为 DNS 已经把 `myagent-lab.online` 解析到 `43.129.193.131` 这个公网 IP,域名走的是同一条链路。我们注意到三件事:① 地址栏挂着"不安全"标签(IP 和域名两种方式都标);② URL 后面挂着 `:12233` 这种自选端口号(正经网站走 80 / 443 标准端口,URL 后面什么都不挂);③ 上面 F12 那个 Network 标签里所有请求的响应体都是**明文 JSON**——任何中间网络节点都能看到甚至改掉。前两件是浏览器层面的视觉提醒,第三件才是底层流量的安全隐患。这三件事之后我们使用 nginx + HTTPS 之后会一并解决。

&emsp;&emsp;本课业务路由全部公开,**直接 curl 即可**。<font color=red>**真实生产环境一定需要鉴权**</font>——避免别人扫到接口随意调用(尤其 LLM 类接口会烧你的 token 配额),常见做法是 nginx Basic Auth、IP 白名单、Cloudflare Access、JWT 中选一种。

```bash
# 任意业务路由直接 curl,本课不做鉴权
curl http://127.0.0.1:12234/todos
# 期望:空列表 [] 或已有 todo 列表 + HTTP 200
```

&emsp;&emsp;curl 都通过、浏览器看到前端能跟后端对话——意味着公网到服务器的链路彻底打通了。这一刻是个"小里程碑"，但**不是终点**：现实里我们不会让应用直接监听公网（暴露面太大、缺乏 TLS、没有限流），下一章 systemd 接手后我们仍然先用这个 12234 监听。

> <font size=2>【名词解释】<b><font color=red>TLS</font>(Transport Layer Security，传输层安全协议)</b>:HTTPS 底层的加密协议,前身是已废弃的 SSL。在 TCP 之上提供加密、完整性、身份认证三件事,是浏览器跟服务器安全通信的基础。第四章我们会推导 TLS 卸载为什么交给 nginx 做。</font>

&emsp;&emsp;ai-todo 在服务器跑起来了。**从"本机 localhost 调通"到"VPS 上前端 12233 + 后端 12234 双端口公网可访问 + 浏览器看到前端 UI"——这一章我们让 ai-todo 第一次走出本机。** 但留下两个尴尬问题——前端那个 `python -m http.server` 是临时演示用的、SSH 一断就挂了；后端 uvicorn 同样是手动起的、终端一关也挂了。下一章我们让 systemd 接管后端进程（前端那个临时 web server 第四章 nginx 上线后会被取代，不用 systemd 守），让 ai-todo 自己活下来。

---

## <center>第三章 systemd 守护进程</center>

&emsp;&emsp;上一章 uvicorn 已经在公网上跑起来了,但我们用的是手动 SSH 上去前台起的命令——SSH 一断服务就挂了、服务器一重启它也起不来、进程崩了更没人会回来重启。这只能算"连通验证",不能算"上线"。这一章我们把 ai-todo 进程交给 systemd 守,让它具备工业部署的三件基本能力:崩了自动重启、reboot 自启、SSH 断开互不影响。

&emsp;&emsp;systemd 在这个角色里要做三件事:进程崩了能自动拉起、机器 reboot 之后能自启、SSH 会话断开不影响后台进程。配上之后,我们用 `journalctl -fu ai-todo` 就能随时看 ai-todo 的滚动日志。

<div align=center><font size=2 color=#999999>systemd 守护 ai-todo 进程的三件能力</font></div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/agent-deploy/2026-05-15/L4-systemd-guard-ecbec470.png" width=80% alt="systemd 守护 ai-todo 进程的三件能力"></div>

<br>

&emsp;&emsp;上一章我们手动跑 uvicorn 解决了"HTTP 通了"这件事，但同时留下了三个隐患：SSH 断了进程死、reboot 后没人启、服务如果崩了没人重新拉起来。这三件事是工业部署的基线要求，我们用 **systemd** 一次性解决——systemd 是 Linux 自带的系统服务管理器(Ubuntu 24.04 用的版本是 255),专门干"守护后台服务"这种活,是工业界跑长期进程的事实标准。这一章我们先拆开 systemd 的组成(unit 文件 + systemctl + journal),再逐字写一份完整的 unit 文件,启用 enable 让它开机自启,最后用 `kill -9` 验证 systemd 是不是真的能把它自动拉起来。

### 3.1 systemd 由什么组成

&emsp;&emsp;先说 systemd 这个东西本身。**systemd 是 Linux 启动后第一个用户态进程——PID 1**。内核 boot 完成后,kernel 拉起的第一个进程就是 systemd,所有其它进程(包括 SSH、bash、我们的 gunicorn 和 nginx)都是它的子孙。这就是它能干"开机拉起服务、崩了自动重启、日志统一接管"这些事的物理基础:作为所有进程的祖先,它有合法权力守护和监管下游任何一个。

&emsp;&emsp;systemd 不是单个工具,是一套体系——由三个部件组成:**unit 文件**(描述服务的配置文件)、**systemctl**(管控命令)、**journal**(日志接管)。先认清这三个零件,后面动手写、跑、看日志才不会绕。

<div align=center><font size=2 color=#999999>systemd 整体组成：systemctl 命令、unit 文件、journal 日志三位一体</font></div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/agent-deploy/2026-05-15/L4-systemd-anatomy-dd5843b5.png" width=80% alt="systemd 由 systemctl 命令 + unit 文件 + journal 日志组成的整体结构"></div>

<br>

&emsp;&emsp;**unit 文件**就是一个 INI 风格的纯文本文件,描述一个服务怎么跑。文件名以 `.service` 结尾(`.service` 是 service 类型 unit；systemd 还支持 `.timer` 定时任务、`.socket` 套接字激活、`.target` 启动目标等其它类型,本课只用 `.service`),路径约定有两条:

- `/etc/systemd/system/xxx.service` — **管理员自定义**的服务放这里(本课的 ai-todo.service 就放这里)

- `/lib/systemd/system/xxx.service` — **系统包管理器**(apt install nginx 等)自动安装的服务放这里

&emsp;&emsp;**systemctl** 是跟 systemd 交互的主命令——`systemctl start / stop / status / enable / restart` 五件套覆盖了 90% 的日常操作。

&emsp;&emsp;**journalctl** 是看日志的——`journalctl -u ai-todo -f` 滚动看某个服务的日志,stdout/stderr 自动被 systemd 收集,不用我们自己写日志文件。

### 3.2 unit 文件三件套原理

&emsp;&emsp;零件认清了,接下来看 unit 文件里**写什么**。核心是三个字段——`WorkingDirectory`、`EnvironmentFile`、`ExecStart`。这三件跟我们本地手动跑 ai-todo 的三个动作是一一对应的：`cd /home/ubuntu/ai-todo` 对应 `WorkingDirectory`、`source .env`（或者 IDE 里读取 .env）对应 `EnvironmentFile`、`uvicorn main:app ...` 对应 `ExecStart`。理解这个映射，unit 文件就不是"网上抄来的模板"，而是"把我们手敲的命令翻译成 systemd 能读懂的格式"。

<div align=center><font size=2 color=#999999>本地手敲的三条命令 → unit 文件三个字段的一一对应映射</font></div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/agent-deploy/2026-05-15/L4-unit-mapping-13d5cc1c.png" width=80% alt="本地三条命令翻译成 unit 文件三字段的一一对应"></div>

<br>

### 3.3 写完整 unit 文件

&emsp;&emsp;在直接给完整 unit 之前，先交代清楚一个工具切换——**这一节启动命令用 `gunicorn -k uvicorn_worker.UvicornWorker` 而不是上一章那个直接调 `uvicorn` 的形态**。原因是 gunicorn 是 Python 生产级进程管理器，专门为多 worker prefork 模型设计，在三件事上比直接调 uvicorn 更稳：worker 生命周期管理（优雅退出、超时回收、worker 死亡自动补充）、跟 systemd 协作（gunicorn master 是单一前台进程，systemd Type=exec 能干净接管）、生产部署语义清晰（"开发跑 uvicorn、生产跑 gunicorn 套 uvicorn worker"是 FastAPI 生态的事实标准）。

&emsp;&emsp;worker class 来自独立的 `uvicorn-worker` 包(uvicorn 0.30.0 以后把这个类挪出主包了),所以服务器装依赖时要顺手 `pip install uvicorn-worker`。注意本章启动命令仍然 `--bind 0.0.0.0:12234`,host 用 `0.0.0.0` 对公网开放、端口跟 .env 默认一致——这是当前的中间配置,第四章 nginx 装好后我们只改 host、端口不动。

```bash
# 服务器：装 gunicorn 和 uvicorn-worker（不在 requirements.txt 里的运行时依赖）
cd /home/ubuntu/ai-todo
.venv/bin/pip install gunicorn uvicorn-worker
.venv/bin/gunicorn --version    # 期望打印 gunicorn 版本号
```

&emsp;&emsp;gunicorn 和 uvicorn-worker 装好之后，下一步是把这个二进制告诉 systemd——把 unit 文件写进 `/etc/systemd/system/ai-todo.service`，里面 `ExecStart=` 那一行就是调用我们刚装好的 `gunicorn`。下面这条命令用 `sudo tee` 配合 heredoc 一次性写入 unit 文件，复制整段直接粘到 VPS 终端就能落盘；注意 `ExecStart=/home/ubuntu/ai-todo/.venv/bin/gunicorn ...` 这一行的绝对路径跟上面 pip 装的位置完全对应。

&emsp;&emsp;unit 文件分三段:`[Unit]` 写元数据和依赖(这个服务是啥、依赖谁、等谁先起来),`[Service]` 写"怎么跑"(启动命令、环境变量、日志、重启策略全在这段,是 unit 文件的核心),`[Install]` 写"开机自启时挂到哪"(`WantedBy=multi-user.target` 表示开机进入多用户模式时自动拉起这个服务)。

```bash
# 一次性把 unit 文件写入 /etc/systemd/system/ai-todo.service
sudo tee /etc/systemd/system/ai-todo.service > /dev/null <<'EOF'
[Unit]
Description=AI Todo FastAPI Service
After=network-online.target
Wants=network-online.target

[Service]
Type=exec
User=ubuntu
Group=ubuntu
WorkingDirectory=/home/ubuntu/ai-todo
EnvironmentFile=/home/ubuntu/ai-todo/.env
Environment="PYTHONUNBUFFERED=1"

# 工业版：gunicorn + UvicornWorker，推荐
ExecStart=/home/ubuntu/ai-todo/.venv/bin/gunicorn main:app \
    -k uvicorn_worker.UvicornWorker \
    --workers 5 \
    --bind 0.0.0.0:12234 \
    --timeout 120 \
    --graceful-timeout 30 \
    --keep-alive 5

Restart=on-failure
RestartSec=5
StartLimitBurst=5
StartLimitIntervalSec=60s

StandardOutput=journal
StandardError=journal
SyslogIdentifier=ai-todo

[Install]
WantedBy=multi-user.target
EOF
```

&emsp;&emsp;下面我们把 `[Service]` 段里几个关键字段单独点一下:

- **`Type=exec`**:systemd 等到新进程映像真正加载完成才把服务标 `active`。默认的 `Type=simple` 只要 fork 成功就立刻标 active,启动命令路径写错、venv 不存在的时候也会给假绿灯。本课全程用 `Type=exec` 拿到更准的状态信号。


- **`Restart=on-failure` + `StartLimitBurst=5` + `StartLimitIntervalSec=60s`**:非零退出码就自动重启,但 60 秒内最多重启 5 次,超过就放弃——防止应用反复崩溃把 CPU 烧光。


- **`StandardOutput=journal`**:把应用 stdout / stderr 都接到 systemd journal,后面 `journalctl -fu ai-todo` 就能看到完整日志,不用单独再配应用层 log 文件。


- **`--workers 5`**:gunicorn 官方推荐公式是 `workers = (2 × CPU 核心数) + 1`——本课用的腾讯云 2 核轻量服务器算出来是 5,直接按公式填。


- **`--log-config` 这类引用外部文件的 flag**:gunicorn / uvicorn 命令行里有些 flag 指向配置文件(譬如 `--log-config /path/to/log.json`),如果文件不存在,systemd 启动时直接报 `Path ... does not exist`,整个服务起不来。本课启动命令不引入额外的日志配置文件,让 gunicorn 走默认行为就够了。

### 3.4 启用 + 验证自动重启

&emsp;&emsp;unit 文件落到 `/etc/systemd/system/ai-todo.service` 之后，跑下面这一组命令把它启用、看实时日志、再验证自动重启。

```bash
# 启用 ai-todo 服务
sudo systemctl daemon-reload                # 重读 /etc/systemd/system/ 下所有 unit 文件,新增或改过的 unit 必须先 reload
sudo systemctl enable --now ai-todo         # enable=开机自启,--now=顺手立刻起来一次,一条命令两件事
sudo systemctl status ai-todo --no-pager    # 看当前状态,期望 Active: active (running);--no-pager 让输出直接打出来不进 less

# 看实时日志
journalctl -fu ai-todo                      # -f 跟随新日志,-u 限定到 ai-todo 这个 unit;Ctrl+C 退出
```

&emsp;&emsp;然后验证自动重启机制——我们手动 `kill -9` 把 gunicorn 主进程干掉，看 systemd 会不会按 `Restart=on-failure` 把它拉起来。

```bash
# 自动重启验证：kill -9 后看 systemd 拉起
sudo pkill -9 -f gunicorn
sleep 6
sudo systemctl status ai-todo --no-pager   # 期望仍然 Active: active，主 PID 已变
```

&emsp;&emsp;`systemctl status` 仍然显示 active，但主 PID 是新的——这意味着旧进程死掉之后 systemd 在 `RestartSec=5` 秒后拉起了一个新的 gunicorn 实例，覆盖了所有应用层崩溃的场景。reboot 自启不需要单独测——`systemctl enable --now ai-todo` 这一行里的 `enable` 已经把开机自启配好了。

&emsp;&emsp;至此后端进程交给 systemd 守着——**崩了自动重启、reboot 自启、日志归 journald**,这一章的目标就达成了。但前端那个 `python -m http.server` 还在 SSH 里临时跑、端口奇怪、Chrome 还在地址栏标红"不安全",下一章我们用 nginx 一并解决,顺手把 `--bind 0.0.0.0:12234` 收回到 loopback。

---

## <center>第四章 nginx 入口：从 HTTPS 推导到生产级扩展</center>

&emsp;&emsp;前面我们让后端 gunicorn(uvicorn worker)在 `0.0.0.0:12234` 上跑着、systemd 把它守了起来,前端用 `python -m http.server` 临时跑在 `12233`——HTTP 通了、进程稳了、SSE 流式 token 也直出浏览器,UI 能看到。**第二章浏览器访问那一段,我们用 IP 和域名两种方式都打开过前端,你应该注意到地址栏挂着一个红色"不安全"标签**——这是 Chrome 在告诉我们:走的是 HTTP 明文,任何中间网络节点都能看到甚至改掉流量。这一章要做的事情就是把 HTTPS 接上,让地址栏的"不安全"变成“链接是安全的“。

&emsp;&emsp;这件事我们用 **nginx** 来做。接下来三节顺着"为什么 → 最小可用 → 生产级"展开:**第一节**推导 HTTPS 到底跟 HTTP 差什么、为什么必须用;**第二节**装一个最小可用的 nginx + HTTPS,让浏览器看到那把小锁;**第三节**把 nginx 升级到生产级配置,顺手补上几件 nginx 默认配不上的事。每一节遇到新概念时再现场解释,这里先不堆术语。

<div align=center><font size=2 color=#999999>nginx 作为唯一入口承担 TLS + SSE + 限流三件事</font></div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/agent-deploy/2026-05-15/L5-nginx-entry-ea0874ed.png" width=80% alt="nginx 作为唯一入口承担 TLS + SSE + 限流三件事"></div>

<br>

### 4.1 为什么需要 HTTPS

&emsp;&emsp;装 nginx 之前先回答一个绕不开的问题——为什么必须上 HTTPS?这一节按"HTTP 哪里不安全 → HTTPS 怎么解决 → 这个方案怎么落到 nginx 上"三步推下来,链路走通了,nginx 才不是"凭感觉装的一个东西",而是被需求一步步推出来的最优解。

<div align=center><font size=2 color=#999999>HTTP 明文 vs HTTPS 加密对比，引出入口工具需求</font></div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/agent-deploy/2026-05-15/L5.1-https-derivation-423ab43c.png" width=80% alt="HTTP 明文 vs HTTPS 加密对比，引出入口工具需求"></div>

<br>

#### 4.1.1 HTTP 不安全：请求和响应都是明信片

&emsp;&emsp;我们打开浏览器访问上一章的 `http://公网IP:12233`,Chrome 地址栏立刻标红"不安全"。点开开发者工具的 Network 标签,前端 JS 跨域调 `http://公网IP:12234` 后端时,请求的 Headers、Request Body、Response Body 全都是明文——浏览器、中间路由器、wifi 接入点、ISP、机房交换机,任何一个中间节点都能原样看到。**明文流量本身就是问题**:你创建的 todo 标题、跟 AI 聊的对话内容、会话 ID 这些数据都在网络上裸奔,任何抓包的中间节点都能完整读出"用户今晚 8 点吃火锅"或者"帮我写一封跟老板请病假的邮件"——生产环境上无论鉴权层做得多严,只要走 HTTP,内容本身就是公开的。

&emsp;&emsp;更危险的是**防篡改**问题。HTTP 不只是明文可读,还是**可改**——中间节点可以在你毫无察觉的情况下改掉响应体。常见场景:运营商劫持往 HTML 里塞广告 JS;咖啡店 wifi 给 fetch 返回的 JSON 注入恶意脚本;公司代理把你的 POST 请求的 body 改成别的内容再发给服务器。HTTP 协议本身没有任何"内容是不是被改过"的校验机制,浏览器和服务器都不知道流量被动过手脚。

&emsp;&emsp;HTTP 协议从 1.0 设计的时候就没有保密性概念——它是明文传输协议。中间任何能抓包的节点(公共 wifi、ISP 出口、被入侵的家用路由器)拿到这些明文都能直接读、直接改。**今年(2026)4 月发布的 Chrome 147 已经对开启了 Enhanced Safe Browsing 的用户默认启用 "Always Use Secure Connections",10 月发布的 Chrome 154 会把这个开关推给所有用户**——届时访问任何公网 HTTP 站点都会先弹一个"该站点不支持安全连接,攻击者可能查看或更改你发送的信息"警告框,要点"继续访问"才能进(出处:Google [security 博客](https://blog.google/security/https-by-defau/) HTTPS by Default 时间表)。业界正在被逼着全面切 HTTPS。

#### 4.1.2 HTTPS 解决了什么

&emsp;&emsp;HTTPS = HTTP + TLS。TLS 这一层做三件事：① **加密**——中间节点看到的都是密文乱码；② **防篡改**——中间节点改动密文，浏览器解密时会检测到 MAC 校验失败；③ **服务器身份证明**——CA 签发的证书证明"这台服务器确实是 myagent-lab.online"，中间人即使拦截了流量也没法伪装。TLS 1.3 握手一般 1 个 RTT 完成，对浏览器用户体验上几乎没有额外延迟。

#### 4.1.3 HTTPS 需要证书 + nginx 完成 TLS 卸载

&emsp;&emsp;4.1.2 里 TLS 的三件事中,"加密"和"防篡改"都由协议本身的密钥协商和 MAC 校验自动完成,不依赖外部组件;但"服务器身份证明"必须有一个**外部信任锚**——光让服务器自称"我是 myagent-lab.online"没用,浏览器凭什么相信?这就是<font color=red>证书</font>要解决的事。CA(浏览器内置信任的可信第三方机构)给域名签发一张证书,内容大致是"我 CA 认证:这把公钥的持有者确实控制 myagent-lab.online 这个域名"。

&emsp;&emsp;这里要澄清一个常见误区——HTTPS 的加密**不是**浏览器提前知道服务器的"密码",而是浏览器和服务器在握手阶段做两件事:① 浏览器用本地内置的 CA 公钥 + 证书里的公钥**验真假**(验证证书的 CA 签名 + 验证服务器是否持有匹配私钥);② 双方各自贡献随机数 + 临时公钥,**临时协商**出一把只有双方知道的会话密钥,这把钥匙从未在网络上传输,中间人抓包也推不出。握手完成后,所有 HTTP 请求和响应都用这把会话密钥**对称加密**——速度快,且只有持有同一把钥匙的双方能解。

<div align=center><font size=2 color=#999999>一次 HTTPS 握手:验证证书 → 协商会话密钥 → 对称加密通信</font></div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/agent-deploy/2026-05-15/L5.1b-tls-handshake-ed433144.png" width=85% alt="HTTPS 握手三段时序:验证证书+协商会话密钥+对称加密通信"></div>

<br>

&emsp;&emsp;那中间人想伪装 myagent-lab.online,直接也从 CA 那里搞一张同名证书不就行了?CA 在签发前会要求申请人**证明自己确实控制这个域名**(下文 certbot 跑的 ACME 流程就是在做这件事)。中间人控制不了 myagent-lab.online 的 DNS 也碰不到这台 VPS,根本拿不到证书;浏览器解密握手时发现证书签名对不上,直接报 `ERR_CERT_AUTHORITY_INVALID` 拒绝建立连接。证书就是这把"身份证明"的钥匙,有 HTTPS 就<font color=red>必须有 CA 签发的证书</font>。

<div align=center><font size=2 color=#999999>证书信任链:CA 签证书要 ACME 域名控制权证明,中间人没控制权拿不到证书</font></div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/agent-deploy/2026-05-15/L5.1-cert-trust-chain-06d7b20e.png" width=80% alt="证书信任链:CA 签证书要 ACME 域名控制权证明,中间人没控制权拿不到证书"></div>

<br>

> <font size=2>【名词解释】<b><font color=red>CA</font>(Certificate Authority，证书颁发机构)</b>:给域名签发 TLS 证书的可信机构。浏览器内置 CA 公钥列表,只信这些 CA 签的证书。Let's Encrypt 是其中之一,免费且支持 ACME 全自动签发。</font>

&emsp;&emsp;Let's Encrypt 是开放且免费的 CA,签发 **DV 证书**(域名验证级别),有效期 90 天,到期前 30 天可以续期。

> <font size=2>【名词解释】<b><font color=red>DV 证书</font>(Domain Validation，域名验证证书)</b>:最常见的证书等级,只验证"申请人能控制这个域名",不验证组织身份。Let's Encrypt 只签 DV,免费可自动续期,对教学和小型服务足够。</font>

&emsp;&emsp;证书要装在"对外入口"这一层——本课直接用 **nginx 作为入口**承担 TLS 卸载:nginx 监听 443 端口,把 HTTPS 流量解开拿到明文 HTTP,再转给后端 gunicorn 监听的 `127.0.0.1:12234`。业界把"解开 HTTPS 拿到明文"这一步叫做 TLS 卸载或 TLS 终止,指的是同一件事。

&emsp;&emsp;选 nginx 的核心理由有三点:

- ① **职责分离**——nginx 专管 TLS,应用进程重启不影响 HTTPS 握手;

- ② **certbot 深度集成**——一条命令搞定签发 + 改配置 + 自动续期 + reload;

- ③ **工业事实标准**——后面要做的限流、SSE 透传、静态资源服务、多应用共用 443 端口,nginx 都是这一层的开箱标配。

> <font size=2>【名词解释】<b><font color=red>反向代理</font>(Reverse Proxy)</b>:站在应用前面替它接外部请求的入口组件,转发请求再把响应回传给客户端。本课 nginx 在 443 对外、转给 loopback 上的 gunicorn,就是典型反向代理。</font>

&emsp;&emsp;顺便提一句:uvicorn 自带 `--ssl-keyfile` / `--ssl-certfile` 参数,技术上也能自己挂证书做 TLS——但工程上几乎不会这么走:应用进程跟 TLS 绑死、证书续期后必须重启应用才能加载、SSL 性能不如 nginx 的 C 实现、限流 / 静态资源 / 多应用共用 443 都得自己写代码。本课不展开这条路径。

&emsp;&emsp;需求链路推清楚了。下一节 nginx 正式登场,我们要做的事是:装 nginx、写最小反代让 80 端口能用、把后端 gunicorn 从 `0.0.0.0:12234` 收回到 `127.0.0.1:12234`(只改 host 不改端口)、kill 临时前端 web server、跑 certbot 一键签证、浏览器以 HTTPS 打开 ai-todo——本课最大里程碑。

### 4.2 nginx 反代登场

&emsp;&emsp;上一节我们把需求链路推清楚了——HTTPS 必须上、证书要装在入口、入口工具选 nginx。这一节我们正式动手装,跑通最小反代,把后端 gunicorn 从 `0.0.0.0:12234` 收回到 `127.0.0.1:12234`(只改 host 不改端口),顺手 kill 临时前端 web server,最后用 certbot 一键签证让浏览器以 HTTPS 打开 ai-todo——这是本课最大的里程碑。

&emsp;&emsp;这一节走四步:装 nginx → 最小反代 → 后端收回 loopback → certbot 签证。每一步走完都有一个可 curl 验证的中间态,任何一步翻车都能精确定位到那一步而不是回头排整条链路。

<div align=center><font size=2 color=#999999>装 nginx → 反代通 → 收回 loopback → 加锁四步流程</font></div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/agent-deploy/2026-05-15/L5.2-nginx-launch-f4ba9299.png" width=80% alt="装 nginx → 反代通 → 收回 loopback → 加锁四步流程"></div>

<br>

&emsp;&emsp;这一节走 5 步实操：装 nginx 摸默认结构 → 写最小反代 server 块 → 改 systemd 把后端收回 loopback → certbot 一键签证 → 浏览器以 HTTPS 打开 ai-todo。每一步结束都有一个可 curl 验证的产物。

#### 4.2.1 装 nginx + 摸默认目录结构

&emsp;&emsp;Ubuntu 24.04 apt 源里 nginx 的版本是 1.24，我们直接装。装完之后浏览器打开 `http://公网IP` 会看到 nginx 默认的 "Welcome to nginx" 欢迎页，这表示 nginx 起来了、80 端口在监听。

```bash
# 装 nginx 并摸目录结构
sudo apt install -y nginx
ls /etc/nginx/sites-available /etc/nginx/sites-enabled
# 默认状态：sites-available/default + sites-enabled/default 软链
```

> <font size=2>【名词解释】<b><font color=red>软链</font>(symbolic link,符号链接)</b> — Linux 文件系统的"快捷方式",`ln -s 真实文件 链接名` 创建一个指向真实文件的引用,删链接不影响本体,`ls -l` 看到 `default -> /etc/nginx/sites-available/default` 这种箭头标记的就是软链。</font>

&emsp;&emsp;Debian 和 Ubuntu 系的 nginx 包采用 `sites-available` + `sites-enabled` 的软链模式来管理多个虚拟主机配置,先把两个目录的角色记清楚:

- `sites-available/` — **配置仓库,存本体**。所有可用的 site config(无论启用与否)都放这里,这才是真正的配置文件,改配置就在这里改

- `sites-enabled/` — **启用清单,只存软链**。目录里没有本体,每个文件都是软链回指 `sites-available/` 里的同名本体,几十字节的指针。`ls -l /etc/nginx/sites-enabled/` 会看到 `myagent-lab.online -> /etc/nginx/sites-available/myagent-lab.online` 这种箭头标记,就是软链特征

&emsp;&emsp;启用某个 site 就 `ln -s ../sites-available/{site} ./{site}` 出一条软链到 `sites-enabled/`,停用就 `rm` 掉软链——本体留在仓库里随时可重启用,不用复制粘贴也不会丢配置。nginx 主配置 `/etc/nginx/nginx.conf` 末尾有一行 `include /etc/nginx/sites-enabled/*;`,启动时把所有启用清单里的 site 包进来生效。

&emsp;&emsp;**为啥用软链而不是直接把配置文件复制到 `sites-enabled/`?** 三个核心好处:

- **单一真相**——配置只有 `sites-available/` 一份本体,改完立即对启用的 site 生效,不会出现"启用副本和仓库副本不一致"

- **启停原子化、可逆**——`ln -s` 启用 / `rm` 停用,只动指针不动内容。复制模式下停用要删整份配置,恢复时还得手敲重写或翻备份

- **本体不丢**——停用时只删指针,配置本体永远留在仓库,`ln -s` 一行就能恢复

&emsp;&emsp;轻量(软链几十字节)只是附带的小好处,主要价值在上面三条——这套设计真正解决的是配置一致性和启停可逆。

#### 4.2.2 写最小 server 块 + 软链启用

&emsp;&emsp;在动手之前先解释一下**为啥要停用 default**——你现在 `ls /etc/nginx/sites-enabled/` 看到的 `default` 是 Ubuntu nginx 包预装的兜底 site,里面写了 `listen 80 default_server` + `server_name _`,意思是"80 端口的兜底响应、匹配任何域名"。你刚才在浏览器打开公网 IP 或 myagent-lab.online 看到的那张 "Welcome to nginx!" 欢迎页,就是这个 default site 渲染的——根目录指向 `/var/www/html/index.html`,这张页是 Ubuntu nginx 包自带的静态文件。如果不停用它,我们新建的 myagent-lab.online site 也想监听 80 端口,两个 server 块抢同一个端口配置会冲突,nginx reload 直接报错。停用 default 后,80 端口空出来给我们的 site 接管,刷新浏览器就能看到 ai-todo 的接口响应而不是这张欢迎页。停用方式按上一节说的——只删 `sites-enabled/default` 这条软链,本体 `sites-available/default` 留在仓库里随时可恢复。

&emsp;&emsp;接下来三步:停用 default、新建 myagent-lab.online 的 site config、再软链启用。最小 server 块只需要三件:`listen 80`、`server_name`、一个 `location /` 把请求 `proxy_pass` 给后端。

&emsp;&emsp;**这里要交代清楚一个时序问题**——这一刻后端还监听在 `0.0.0.0:12234`(上一章留下的中间配置,host 是 `0.0.0.0` 对公网开放),但我们 nginx 这边的 site config <b>直接写最终态的 `127.0.0.1:12234`</b>。这样写是有意的:nginx 配置和后端 bind 改动是两个独立的动作,我们 nginx 这边一次配到位,下一步把后端 host 一次性切到 `127.0.0.1`(端口不变)——两边对齐之后整套就直接到生产形态了。

```bash
# 删默认 site，新建并软链 myagent-lab.online
sudo rm -f /etc/nginx/sites-enabled/default
sudo touch /etc/nginx/sites-available/myagent-lab.online
sudo ln -sf /etc/nginx/sites-available/myagent-lab.online \
            /etc/nginx/sites-enabled/
```

&emsp;&emsp;下面这条命令把最小 server 块一次性写到 `/etc/nginx/sites-available/myagent-lab.online`——复制整段粘到 VPS 终端即可：

```bash
# 把最小 server 块写入 nginx site config
sudo tee /etc/nginx/sites-available/myagent-lab.online > /dev/null <<'EOF'
# 4.2.2 最小版本
server {
    listen 80;
    listen [::]:80;
    server_name myagent-lab.online;

    location / {
        proxy_pass http://127.0.0.1:12234;
        proxy_http_version 1.1;
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
        proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
        proxy_set_header X-Forwarded-Proto $scheme;
    }
}
EOF
```

&emsp;&emsp;`proxy_set_header` 的四件是 nginx 反代的标配。`Host` 让后端知道客户端访问的是哪个域名（多虚拟主机场景必要）；`X-Real-IP` 把客户端真实 IP 透给后端（不然后端只能看到 nginx 的 127.0.0.1）；`X-Forwarded-For` 是 IP 链路追溯，每过一层代理就 append 一段；`X-Forwarded-Proto` 让后端知道客户端是用 HTTP 还是 HTTPS 进来的（重要：HTTPS 终止后 nginx 转给后端的是明文 HTTP，后端如果不看这个 header 会以为客户端走的就是 HTTP，搞错 redirect 链路）。

#### 4.2.3 uvicorn 收回 127.0.0.1:12234

&emsp;&emsp;nginx 上一步已经按最终态配好了反代到 `127.0.0.1:12234`，现在做三件事:① 改 systemd unit 把 ai-todo 后端的 host 从 `0.0.0.0` 收回 `127.0.0.1`(端口都是 12234,只改 host);② kill 掉第二章那个临时前端 `python -m http.server 12233`(它的职责马上由 nginx 接管);③ reload nginx、重启 ai-todo。

```bash
# ① 改 systemd unit：host 从 0.0.0.0 收回 127.0.0.1（端口 12234 不变）
sudo sed -i 's|--bind 0.0.0.0:12234|--bind 127.0.0.1:12234|' \
    /etc/systemd/system/ai-todo.service

sudo systemctl daemon-reload
sudo systemctl restart ai-todo

# ② kill 临时前端 web server（nginx 即将接管前端静态托管）
sudo pkill -f "http.server 12233"

# ③ nginx 启用最小反代
sudo nginx -t
sudo systemctl reload nginx
```

&emsp;&emsp;这一步把 ai-todo 从"对公网开放"收回到"只接受来自 nginx 的请求"。`127.0.0.1` 这个地址是 loopback 接口，数据包在内核里就被路由回本机，不会出物理网卡——公网从外面 curl 12234 端口会 timeout，因为根本到不了这台机器的 12234 监听套接字。临时前端那个 12233 端口同样不再监听。我们 curl 三发验证：

```bash
# 验证职责分离（在服务器或自己电脑跑都行）
curl -i http://myagent-lab.online/health              # 期望 200，经 nginx 反代
curl -m 3 http://43.129.193.131:12233/                # 期望 connection refused（前端临时 web server 已停）
curl -m 3 http://43.129.193.131:12234/health          # 期望 timeout（后端收回 loopback，公网不出网卡）

# Windows PowerShell：curl 改成 curl.exe
```

&emsp;&emsp;三个 curl 结果对应三个事实:① 经 nginx 80 端口反代到 loopback 12234 通了;② 第二章那个临时前端 `python -m http.server 12233` 已经退役(前端职责马上由 nginx 接管);③ 12234 是 loopback 端口,公网根本到不了。从这一刻开始,ai-todo 的对外入口只剩 nginx——符合"nginx 是唯一入口"的目标拓扑,第三章 unit 文件里 `--bind 0.0.0.0:12234` 那行也在这一节切到了最终态。

#### 4.2.4 certbot 一键签证：snap 渠道 + --nginx 自动改配置

&emsp;&emsp;前面讲过 HTTPS 必须有 CA 签发的证书,但手动完成整套流程(跟 Let's Encrypt 跑 ACME 协议、证明域名所有权、下载证书、改 nginx 配置加 `ssl_*` 行、配 80→443 重定向、设置 90 天到期自动续期)要敲十几步命令,容易出错。certbot 把这套流程压成一条命令——告诉它"给 myagent-lab.online 签个证书 + 改 nginx 配置 + 加 80→443 重定向",剩下全自动。

> <font size=2>【名词解释】<b><font color=red>certbot</font>(certificate bot,证书机器人)</b> — Let's Encrypt 官方维护的 ACME 协议客户端,跟 CA 通信、证明域名所有权、下载证书、改 nginx 配置、设置 systemd timer 自动续期这一整套动作都由它完成,本课用 `certbot --nginx -d {域名} --redirect` 一条命令搞定签发 + 配置 + 重定向。</font>

&emsp;&emsp;Ubuntu 24.04 官方推荐 certbot 走 snap 渠道，<font color=red>**不要装 apt 版的 `python3-certbot-nginx`**</font>——apt 版长期不更新，跟 snap 版会冲突。snap 装好后做一个软链让 `certbot` 在 PATH 里能找到。

```bash
# certbot：snap 渠道安装
sudo snap install --classic certbot
sudo ln -sf /snap/bin/certbot /usr/bin/certbot

# 如果以前装过 apt 版，先卸掉避免冲突
sudo apt remove -y certbot python3-certbot-nginx || true

# 一键签证 + 自动改 nginx 配置 + 配 80→443 重定向
sudo certbot --nginx -d myagent-lab.online --redirect \
    --agree-tos --register-unsafely-without-email
```

&emsp;&emsp;这里的 `--register-unsafely-without-email` 参数让 ACME 注册不绑邮箱，可以免去交互输入很方便——代价是 Let's Encrypt 这边没有我们的邮箱，证书快到期、CA 有安全公告时收不到提醒邮件。正式生产建议去掉这个参数、改成 `--email your@email.com` 把真实邮箱绑上，多一道保险（即便 certbot snap timer 自动续期一直工作正常，邮件提醒也是兜底渠道之一）。

<div align=center><font size=2 color=#999999>certbot 一条命令展开 5 件事:跟 CA 协商 → 域名证明 → 证书落盘 → 改 nginx → 自动续期</font></div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/agent-deploy/2026-05-15/L5.4-certbot-pipeline-f2758128.png" width=85% alt="certbot 一条命令展开 5 件事流程图"></div>

<br>

&emsp;&emsp;`certbot --nginx` 这条命令做了几件事：跟 Let's Encrypt 走 ACME 协议申请证书（用 HTTP-01 挑战，nginx 80 端口能通就能验证域名所有权）；自动把证书放到 `/etc/letsencrypt/live/myagent-lab.online/`；自动改我们的 nginx site config 注入 4 行 `ssl_*` 配置 + 1 个 80→443 重定向 server 块。我们 grep 一下看 certbot 改了哪几行：

```bash
# 看 certbot 给 site config 注入了什么
sudo grep -n "managed by Certbot" \
    /etc/nginx/sites-available/myagent-lab.online
```

&emsp;&emsp;每一行 certbot 注入的配置都带 `# managed by Certbot` 注释，意思是"这一行是 certbot 自动管理的，请勿手改"。下次 certbot 续期或更新配置，它会按这些注释找到自己写的内容做替换。

#### 4.2.5 dry-run 续期 + 浏览器以 HTTPS 打开 ai-todo

&emsp;&emsp;Let's Encrypt 证书 90 天有效期,到期前 30 天可以续期。certbot snap 装好时已经自动配了 systemd timer,每天定时检查、需要续期时自动续。但**我们不能等 90 天后真续期出问题才发现 nginx 配错或防火墙挡**——`--dry-run` 就是 Let's Encrypt 专门提供的验收检查模式,把续期全流程跑一遍,但最后不签真证书。

&emsp;&emsp;dry-run 跟真签发最关键的区别是:dry-run 连的是 Let's Encrypt 的 **staging(预发)** 服务器、生成测试证书(浏览器不信任,不会替换 live 目录里的真证书),**不消耗签发配额**;真签发连生产服务器、消耗每域名每周 5 张的配额。每装完 certbot 都应该跑一次 dry-run——提前 90 天暴露所有问题(账户态 / HTTP-01 挑战 / DNS / 证书路径),不会到期时才发现挂了。

```bash
# 续期 dry-run（不消耗配额）
sudo certbot renew --dry-run

# 看自动续期 timer 是否已经排上
systemctl list-timers | grep certbot
# 期望看到 snap.certbot.renew.timer

# HTTPS 端到端验证
curl -i https://myagent-lab.online/health         # 期望 200
# 浏览器打开 https://myagent-lab.online → 地址栏锁标 + 看到 {"detail":"未找到"} JSON
# (根路径 404 是本阶段预期 —— 4.3 配前端静态分流后才会显示 ai-todo 界面)
```

&emsp;&emsp;跑完 `certbot renew --dry-run` 后,关键看输出最后两行 `Congratulations, all simulated renewals succeeded` 和 `/etc/letsencrypt/live/myagent-lab.online/fullchain.pem (success)`——出现这两行表示三件事都验证通过:① Let's Encrypt 账户态正常;② HTTP-01 挑战能完成(nginx 80 端口能接 + `.well-known/acme-challenge` 路径可达 + 域名 DNS 仍指向这台 VPS);③ 证书路径和续期配置文件 certbot 都能找到。看到这条就放心了——90 天里不用人工干预,systemd timer 会自动跑真续期,过程对你完全透明。如果看到的不是 succeeded 而是 failed,日志在 `/var/log/letsencrypt/letsencrypt.log`,先 `cat` 这个文件再排查。

&emsp;&emsp;`/health` 通了只验证了一条路由。我们顺手跑一条业务路由 `/todos`,把 HTTPS → nginx → loopback → 后端这条全链路也走一遍(本课不做鉴权,直接 curl 即可):

```bash
# macOS / Linux / Git Bash:业务路由经 HTTPS + nginx + loopback 链路,直接调
curl https://myagent-lab.online/todos
# 期望:空列表 [] 或已有 todo 列表 + HTTP 200
```

```bash
# Windows PowerShell:用 curl.exe(不是 PowerShell alias)
curl.exe https://myagent-lab.online/todos
```

&emsp;&emsp;到这一步,命令行 `curl -I https://myagent-lab.online/health` 拿到 200、`/todos` 拿到 200,说明 HTTPS 握手 / nginx 反代 / loopback 隔离 / 后端接到请求这条链路通了。打开浏览器访问 `https://myagent-lab.online` 顺手看一眼,**地址栏出现锁标**——这就是 HTTPS 通了、证书被本机 CA 根库信任的最直观信号。

> ⚠️ **本阶段现状**:浏览器访问根路径 `https://myagent-lab.online/` 看到的是 <b>{"detail":"Not Found"}</b>(后端返的 404 JSON)——这是正常的,不是出错。现在 nginx 是上一小节写的最小反代,所有请求都 proxy 到后端 uvicorn,后端没注册 `/` 路由所以 404。**前端要等下一节配完 nginx 三路分流才会上岗**,到时浏览器根路径才能看到 ai-todo 界面。本节我们交付的是"安全可访问的 API 层",前端是下一节的事。

&emsp;&emsp;curl 通 + 浏览器锁标亮 = **本课最大里程碑**。从这一刻起,ai-todo 具备**公网可访问 + HTTPS 加密 + loopback 后端隔离 + 自动续期** —— 入口、加密、隔离、续期这四件事都齐了,是一台合格的公网后端服务。我们一路走过来的所有组件——基线包、代码、systemd、nginx、loopback、证书——在这一刻串成了一条线。

> <font size=2>【名词解释】<b><font color=red>certbot</font>(Let's Encrypt 证书自动化工具)</b>:Let's Encrypt 官方的证书签发和续期工具。一条命令完成域名验证、本机落证书、改 nginx 启用 HTTPS,并装上 systemd timer 自动续期。</font>

&emsp;&emsp;HTTPS 通了,但浏览器打开还看不到 ai-todo 界面——前端 HTML/JS 静态文件需要 nginx 接管 serve(目前 nginx 把所有请求都 proxy 给后端,后端不返前端文件)。下一节我们**先让 nginx 把 frontend/ 目录服务起来**(顺手对比一下"直接起 `python -m http.server 12233`"跟"nginx serve 静态"的优劣),让浏览器能真正看到界面、点开聊天试,然后再加 SSE 透传 + 限流,把入口升级到生产级。

### 4.3 nginx 扩展能力

&emsp;&emsp;HTTPS 接通之后,nginx 只是"能跑通"——浏览器打开 `https://myagent-lab.online` 还看不到 ai-todo 界面(根路径只返 404 JSON,前端 HTML/JS 还没被 nginx 接管 serve)。这一节按"问题驱动"的节奏把 nginx 从最小反代升级到生产级,分两步走:**先让 nginx 接管前端静态 + SSE 端点透传**(浏览器看到界面、SSE 实时蹦字、HTTPS 同源,业务路由也走 443 同入口)→ **再叠加一层限流和防爬**(抗速率攻击 + 屏蔽 `.git` `.env` 这类敏感路径扫描)。

<div align=center><font size=2 color=#999999>nginx 三类路由分发与两个易踩点</font></div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/agent-deploy/2026-05-15/L5.3-nginx-extend-1a046e4c.png" width=80% alt="nginx 三类路由分发与两个易踩点"></div>

<br>

&emsp;&emsp;nginx 入口按三条路由分流:`/chat/events` 和 `/chat/stream` 两个流式端点用 `location =` 精确匹配走 SSE 透传配置(关默认缓冲让 token 不积压、长连接维持 24h)、`/sessions` / `/todos` / `/health` 业务路由用 `location ~` 正则匹配反代给后端 uvicorn、前端 `/` 用 `location /` 兜底直 serve 静态文件。下面 site config 里我们一次到位写完这三组 location 的基础生产版,限流和防爬下一节再叠加。

&emsp;&emsp;先动手。

#### 4.3.1 nginx 接管前端静态:替代 12233 临时 server

&emsp;&emsp;在动手前先回答一个学员常会想的问题——**能不能不让 nginx 接管前端,继续用第 2 节那种 `python -m http.server 12233` 起前端,搭配现在 nginx 443 的 HTTPS 后端?** 技术上**能跑通**,但要绕三个坑、改两处代码,而且最后还是要挪到 nginx,不划算:

- **坑 1:`BACKEND` 常量算出来不对**。前端 `app.js` L21 是 `` const BACKEND = `${location.protocol}//${location.hostname}:12234` ``,浏览器打开 `http://公网IP:12233` 会算出 `BACKEND = "http://公网IP:12234"`——但 12234 已经在前面"收回 loopback"那节挡在公网外面,根本访问不到,要手动改成 `"https://myagent-lab.online"`

- **坑 2:CORS 跨域**。改完 BACKEND 后,前端 origin `http://公网IP:12233` 跟后端 origin `https://myagent-lab.online` 不同,触发 CORS preflight,后端 `cors_allow_origins` 要显式放行——生产场景里这种"前后端不同 origin"本来就不推荐

- **坑 3:浏览器地址栏标红"不安全"**。前端走 HTTP、后端走 HTTPS,用户看到不安全标记还以为整套都没加密,体验混乱

&emsp;&emsp;最关键的是:**就算把上面三个坑都绕了,最终还是要把前端挪到 nginx 443 同源**(让地址栏统一显示锁标 + 前端 JS 同源调 API 不用过 CORS)——绕一圈回到原点。所以正解就是这一节做的事:**让 nginx 同时接管前端静态 + 后端反代,443 统一入口**。

&emsp;&emsp;nginx 比 `python -m http.server` 在 4 个维度上彻底碾压,**不是被迫替换,是更优**:

| 维度 | `python -m http.server 12233` | nginx serve 静态 |
|---|---|---|
| **进程模型** | 单进程同步,一个慢请求就卡住其他人 | 多 worker + 异步 epoll,几千并发不卡 |
| **守护** | 前台进程,SSH 一断就死,要靠 nohup / tmux 续命 | systemd 守护,开机自启 + 崩了自动拉 |
| **HTTPS** | 完全不支持(裸 HTTP) | 配证书一次到位,统一走 443 |
| **优化** | 无 gzip、无缓存控制、无 Range 请求 | 内置 gzip 压缩、`Cache-Control`、`ETag`、断点续传 |

&emsp;&emsp;Python 标准库 `http.server` 文档第一行就明确写"not recommended for production"——它是**开发临时验证**用的玩具级 web server,nginx 是**生产级 Web 服务器和反向代理**。本课从第 2 节的临时方案过渡到 nginx 接管,正是把"能跑通"升级成"能上线"的关键一步。

&emsp;&emsp;先把 ai-todo 真实路由清单挂这里,后面写 site config 要按这套真实前缀分流:`/chat/stream`(POST 普通流式)、`/chat/events`(GET 多事件 SSE)、`/sessions/*`(会话管理)、`/todos/*`(待办 CRUD)、`/health`(健康检查)——<b>没有 `/api/` 统一前缀</b>,所以 nginx 业务正则按真实前缀写,别试 `location /api/` 那种统一兜法。

&emsp;&emsp;一把 `sudo tee` 把 site config 升级到基础生产版——前端静态 alias + 两个 SSE 端点精确匹配走透传 + 业务路由正则反代 + HTTP/2 + 上传体积上限 + 日志路径定制,一次到位:

```bash
# 基础生产 site config
sudo tee /etc/nginx/sites-available/myagent-lab.online > /dev/null <<'EOF'
# 80 端口:certbot 注入的 80→443 重定向(原样保留)
server {
    listen 80;
    server_name myagent-lab.online;
    if ($host = myagent-lab.online) {
        return 301 https://$host$request_uri;   # managed by Certbot
    }
    return 404;
}

# 443 端口:HTTPS 终止 + 反代后端 + serve 静态前端
server {
    listen 443 ssl http2;       # 启用 HTTP/2 多路复用,前端静态资源并发加载更快
    server_name myagent-lab.online;
    client_max_body_size 20M;   # 上传体积上限,防大文件攻击拖垮带宽

    # certbot 注入的 4 行 SSL 配置
    ssl_certificate /etc/letsencrypt/live/myagent-lab.online/fullchain.pem;     # managed by Certbot
    ssl_certificate_key /etc/letsencrypt/live/myagent-lab.online/privkey.pem;   # managed by Certbot
    include /etc/letsencrypt/options-ssl-nginx.conf;                            # managed by Certbot
    ssl_dhparam /etc/letsencrypt/ssl-dhparams.pem;                              # managed by Certbot

    # 日志路径(后面的日志章节会基于这两个 log 跑日志拆解)
    access_log /var/log/nginx/myagent-access.log;
    error_log  /var/log/nginx/myagent-error.log;

    # SSE 端点 1:多事件 SSE,location = 精确匹配优先级最高,绝不被业务正则抢走
    location = /chat/events {
        proxy_pass http://127.0.0.1:12234;
        proxy_http_version 1.1;
        proxy_set_header Connection "";          # 清空 Connection 维持长连接
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
        proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
        proxy_set_header X-Forwarded-Proto $scheme;

        # SSE 三件套:防御性关掉所有可能的缓冲层
        proxy_buffering off;
        gzip off;
        add_header X-Accel-Buffering no always;

        # SSE 长连接:默认 60s 会被踢,加大到 24 小时
        proxy_read_timeout 86400s;
        proxy_send_timeout 86400s;
    }

    # SSE 端点 2:普通流式(StreamingResponse),配置跟 /chat/events 同模板
    location = /chat/stream {
        proxy_pass http://127.0.0.1:12234;
        proxy_http_version 1.1;
        proxy_set_header Connection "";
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
        proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
        proxy_set_header X-Forwarded-Proto $scheme;

        proxy_buffering off;
        gzip off;
        add_header X-Accel-Buffering no always;

        proxy_read_timeout 86400s;
        proxy_send_timeout 86400s;
    }

    # 业务路由:ai-todo 真实前缀正则匹配,proxy 给后端
    location ~ ^/(sessions|todos|health) {
        proxy_pass http://127.0.0.1:12234;
        proxy_http_version 1.1;
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
        proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
        proxy_set_header X-Forwarded-Proto $scheme;
        proxy_connect_timeout 5s;       # 后端连不上 5s 就放弃,不拖死 nginx worker
        proxy_read_timeout 60s;         # 业务接口 60s 应该够
    }

    # 前端静态:把 frontend/ 目录里的文件直接发给浏览器
    location / {
        alias /home/ubuntu/ai-todo/frontend/;
        try_files $uri $uri/ /index.html;
    }
}
EOF
```

&emsp;&emsp;上面这段 heredoc 复制到 `EOF` 闭合行结束就停手,别顺手把下面的 chmod 也选中——粘进 EOF 之前会污染 nginx 配置体。下面是另起的一条命令,给 nginx 开放前端目录的访问权限(下一段会解释为什么必须):

```bash
# 开放 nginx 访问前端目录的权限
sudo chmod o+x /home/ubuntu
sudo chmod -R o+rX /home/ubuntu/ai-todo/frontend
```

&emsp;&emsp;最后校验 nginx 配置语法 + reload 让 site config 生效:

```bash
# 校验 + reload
sudo nginx -t
sudo systemctl reload nginx
```

&emsp;&emsp;先说为什么有 `chmod` 那两行——这是任何按本课流程做的学员都会撞到的坑:**reload 通过但浏览器看到的是 500 Internal Server Error,而不是 ai-todo 界面**,`/var/log/nginx/error.log` 里全是 `stat() "/home/ubuntu/ai-todo/frontend/index.html" failed (13: Permission denied)`。原因是 Ubuntu 上 `/home/ubuntu` 默认权限是 `drwxr-x---`(750),owner 是 `ubuntu` 用户、group 是 `ubuntu` 组,**其他用户(other)完全没权限**。而 nginx 的 worker 进程不是以 `ubuntu` 跑的,而是以系统用户 <font color=red>`www-data`</font> 跑——既不是 owner 也不在 group,被目录权限拒之门外,自然读不到 `frontend/index.html`。所以加一行 `chmod o+x /home/ubuntu` 给 other 加目录穿越权(注意只加 `x` 不加 `r`,nginx 能"穿过去"但不能列出 `/home/ubuntu` 下其他用户目录),再 `chmod -R o+rX /home/ubuntu/ai-todo/frontend` 把前端文件全部对 other 开放读权限(大写 `X` 表示只对目录和已有执行位的文件加 x,普通文件只加 r,不会误把 .css 文件标成可执行)。

> <font size=2>【名词解释】<b><font color=red>www-data</font>(nginx / apache 默认运行用户)</b>:Debian / Ubuntu 系统上 nginx worker 进程的运行身份。`/etc/nginx/nginx.conf` 顶部的 `user www-data;` 就是它。设计上让 web server 用一个<b>最低权限专用账号</b>跑,而不是 root 或某个登录用户——这样万一 web server 被攻破,攻击者拿到的也只是 www-data 权限,进不了别人 home 目录、读不到 `/etc/shadow`。</font>

&emsp;&emsp;权限开通之后浏览器再打开 `https://myagent-lab.online` —— 这次不再是 500,而是 **ai-todo 的真实界面**(左侧会话列表 / 中间聊天框 / 右侧月历 + Todo 列表)。前端 JS 加载完成后自动调 `/sessions` 拉历史、调 `/todos` 拉清单,这些请求被 `location ~ ^/(sessions|todos|health)` 正则匹中 proxy 给 loopback 后端;在中间输入框发消息,POST 走 `/chat/stream` 精确匹配的 SSE 透传 location;切到多事件模式则走 `/chat/events`——四组 location 各司其职。

&emsp;&emsp;site config 里几个关键字段顺手点一下。<b>nginx location 匹配优先级</b>:`location =` 精确匹配优先级最高(级别 1),`location ~` 正则次之(级别 3),`location /` 普通前缀兜底(级别 4)。配置里两个 SSE 端点用 `location =` 是为了**确保它们绝不被业务正则抢走**——否则 `/chat/events` 会先被 `^/(sessions|todos|health)` 之类的正则吃掉(虽然这条没匹中,但实际项目里业务路由命名经常跟 SSE 路径冲突,精确匹配是最稳的隔离方式)。<b>SSE 三件套</b>(`proxy_buffering off` + `gzip off` + `X-Accel-Buffering: no`):Ubuntu 默认 nginx 没开 gzip + FastAPI `StreamingResponse` 用 chunked 主动 flush,所以 SSE 默认就顺畅;但生产环境容易撞坑(同事改全局 `gzip on` / 上 CDN 边缘 buffer / 换发行版默认 buffer on),三件套是生产防御性配置,锁定本 location 不受全局影响。<b>业务超时</b>(`proxy_connect_timeout 5s` + `proxy_read_timeout 60s`):后端连不上 5 秒就放弃,业务接口 60 秒应该够——超过就让 nginx 主动断,不要让 worker 一直挂着等。

&emsp;&emsp;<b>alias vs root</b>:`root` 把 location 前缀<b>加</b>在路径前(`location /foo { root /var/www; }` → 实际路径 `/var/www/foo`);`alias` 把 location 前缀<b>整段替换</b>成 alias 路径(`location /foo/ { alias /var/www/; }` → 实际路径 `/var/www/`)。本课 `location /` 配 `alias /home/ubuntu/ai-todo/frontend/`,URL `/` 映射到 `/home/ubuntu/ai-todo/frontend/index.html`、`/app.js` 映射到 `/home/ubuntu/ai-todo/frontend/app.js`。`try_files $uri $uri/ /index.html` 是 SPA 标准 fallback——先找文件、找不到再找目录、最后兜底返 `index.html` 让前端路由接管(ai-todo 简单 SPA 用不到 router fallback,这一行先备着没副作用)。

#### 4.3.2 限流叠加 + 防爬:抗速率攻击 + 屏蔽敏感扫描

&emsp;&emsp;HTTPS + nginx 入口刚一上线,`/var/log/nginx/error.log` 里就会立刻看到野生扫描器试探敏感路径——典型的一条:

In [ ]:
[crit] *432 stat() "/home/ubuntu/ai-todo/frontend/.git/config" failed (13: Permission denied),
       client: 45.148.10.120, server: myagent-lab.online,
       request: "GET /.git/config HTTP/1.1"

&emsp;&emsp;这是公网每分钟都在发生的事:扫描器把全网 IP 范围撒一遍,挨个试 `/.git/config` / `/.env` / `/.venv/` 这类路径,期望某台服务器没把 git 仓库或环境变量文件挡在静态目录外面。本节给 nginx 入口加两层防御:<b>入口限流</b>(单 IP 每秒最多 5 个请求,SSE 单 IP 最多 2 条并发连接,挡住爬虫和恶意刷接口),<b>防爬路径屏蔽</b>(对 `.git` / `.env` / `.venv` 等敏感路径直接返 404,连"文件存不存在"的信息泄露都不给)。

&emsp;&emsp;限流需要先定义共享内存 zone(放到 `nginx.conf` 的 `http {}` 块,任意 site 都能引用)。我们把 zone 定义落到 `/etc/nginx/conf.d/` 单独一个文件——好处是跟 site config 解耦,以后加新 site 可以直接复用:

```bash
# 限流 zone 定义(http 层共享内存,任意 site 引用)
sudo tee /etc/nginx/conf.d/ai-todo-limits.conf > /dev/null <<'EOF'
limit_req_zone $binary_remote_addr zone=api_rate:10m rate=5r/s;
limit_conn_zone $binary_remote_addr zone=conn_per_ip:10m;
EOF
```

&emsp;&emsp;`api_rate` zone 10MB 共享内存能装大约 16 万个 IP 状态(够单机服务用),`rate=5r/s` 意思是每个 IP 每秒最多 5 个请求,超出按 `burst=10 nodelay` 允许短暂突发到 10 个、再超就回 429。`conn_per_ip` zone 用于 SSE 端点限制每 IP 最多 2 条并发连接,防止恶意客户端开 100 条 SSE 拖死后端。

&emsp;&emsp;接下来覆盖一次 site config,把上一节的基础版升级到带限流和防爬的最终生产版——业务路由叠加 `limit_req`、SSE 端点 1 叠加 `limit_conn` SSE 并发限制、SSE 端点 2 叠加 `limit_req` 速率限制、末尾追加一个 `location ~ /\.(env|git|venv)` 屏蔽敏感路径。其他部分跟上一节完全一致,直接 `sudo tee` 整段覆盖,免得 patch 行号出错:

```bash
# 最终生产 site config(基础 + 限流 + 防爬)
sudo tee /etc/nginx/sites-available/myagent-lab.online > /dev/null <<'EOF'
# 80 端口:certbot 注入的 80→443 重定向(原样保留)
server {
    listen 80;
    server_name myagent-lab.online;
    if ($host = myagent-lab.online) {
        return 301 https://$host$request_uri;   # managed by Certbot
    }
    return 404;
}

# 443 端口:HTTPS 终止 + 反代后端 + serve 静态前端 + 限流 + 防爬
server {
    listen 443 ssl http2;
    server_name myagent-lab.online;
    client_max_body_size 20M;

    ssl_certificate /etc/letsencrypt/live/myagent-lab.online/fullchain.pem;     # managed by Certbot
    ssl_certificate_key /etc/letsencrypt/live/myagent-lab.online/privkey.pem;   # managed by Certbot
    include /etc/letsencrypt/options-ssl-nginx.conf;                            # managed by Certbot
    ssl_dhparam /etc/letsencrypt/ssl-dhparams.pem;                              # managed by Certbot

    access_log /var/log/nginx/myagent-access.log;
    error_log  /var/log/nginx/myagent-error.log;

    # SSE 端点 1:多事件 SSE,叠加单 IP 并发连接数限制
    location = /chat/events {
        limit_conn conn_per_ip 2;       # 单 IP 最多 2 条 SSE 并发
        limit_conn_status 429;

        proxy_pass http://127.0.0.1:12234;
        proxy_http_version 1.1;
        proxy_set_header Connection "";
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
        proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
        proxy_set_header X-Forwarded-Proto $scheme;

        proxy_buffering off;
        gzip off;
        add_header X-Accel-Buffering no always;

        proxy_read_timeout 86400s;
        proxy_send_timeout 86400s;
    }

    # SSE 端点 2:普通流式,叠加速率限流
    location = /chat/stream {
        limit_req zone=api_rate burst=10 nodelay;   # 单 IP 5r/s + burst 10
        limit_req_status 429;

        proxy_pass http://127.0.0.1:12234;
        proxy_http_version 1.1;
        proxy_set_header Connection "";
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
        proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
        proxy_set_header X-Forwarded-Proto $scheme;

        proxy_buffering off;
        gzip off;
        add_header X-Accel-Buffering no always;

        proxy_read_timeout 86400s;
        proxy_send_timeout 86400s;
    }

    # 业务路由:叠加速率限流
    location ~ ^/(sessions|todos|health) {
        limit_req zone=api_rate burst=10 nodelay;
        limit_req_status 429;

        proxy_pass http://127.0.0.1:12234;
        proxy_http_version 1.1;
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
        proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
        proxy_set_header X-Forwarded-Proto $scheme;
        proxy_connect_timeout 5s;
        proxy_read_timeout 60s;
    }

    # 前端静态
    location / {
        alias /home/ubuntu/ai-todo/frontend/;
        try_files $uri $uri/ /index.html;
    }

    # 防爬:屏蔽常见敏感目录扫描(.git / .env / .venv 等)
    location ~ /\.(env|git|venv) {
        deny all;
        return 404;
    }
}
EOF
```

&emsp;&emsp;同上一节,这段 heredoc 复制到 `EOF` 闭合就停手,别把下面 `nginx -t` / `reload` 一起选中粘进去。最后校验 + reload:

```bash
# 校验 + reload
sudo nginx -t
sudo systemctl reload nginx
```

&emsp;&emsp;`nginx -t` 通过、reload 没报错——nginx 入口正式齐活:HTTPS 端到端 + 真实路由分发 + SSE 透传 + 入口限流 + 防爬 + 前端静态兜底,从外面看 ai-todo 是一台合格的生产级公网服务。再观察一会儿 `error.log`,你会持续看到 `45.x.x.x` / `103.x.x.x` 之类的扫描 IP 来试 `/.git/config` / `/.env`——但响应一律是 404,而不是泄露文件内容或 403(那等于告诉对方"路径存在但被禁了")。这就是防爬 location 的实际作用。

> <font size=2>【名词解释】<b><font color=red>limit_req</font>(nginx 请求速率限制指令)</b>:nginx 的请求速率限制指令,用令牌桶按 IP 或自定义 key 控制每秒请求数,超限返回 429。配合 `limit_conn` 限并发,完成入口限流。</font>

&emsp;&emsp;**从"前端 12233 + 后端 12234 双端口 HTTP 裸跑"到"https://域名 + HTTPS 加密 + nginx 统一入口 + 路由分发 + SSE 透传 + 一层限流 + 前端静态托管"——这一章 nginx 入口已经是生产级配置**。下一章我们让日志不爆盘、给 `/health` 配成监控探针入口，把"运维能看见"这一层补全。

---

## <center>第五章 日志轮转、健康检查与监控入口</center>

&emsp;&emsp;nginx 入口已经是生产级配置——HTTPS、SSE、限流齐了。接下来还有几件配套的事要做:日志切分轮转(控制单文件体积 + 定期清理旧的)、状态可探活、故障可追溯。这一章我们把这三件配齐。

&emsp;&emsp;日志三路汇到 logrotate 统一切分压缩;监控这一路把 `/health` 暴露给外部探针。这是上线之后要持续盯着的两条线。

<div align=center><font size=2 color=#999999>三类日志通过 logrotate 轮转 + /health 探针被外部监控接入</font></div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/agent-deploy/2026-05-15/L6-logging-3eebf373.png" width=80% alt="三类日志通过 logrotate 轮转 + /health 探针被外部监控接入"></div>

<br>

&emsp;&emsp;这三件事具体怎么做:**日志切分轮转**——nginx 日志自带一份 logrotate 规则,ai-todo 应用日志手配一份;**状态可探活**——监控工具定期问"你还活着吗",用 ai-todo 暴露的 `/health` 端点;**故障可追溯**——日志能查到哪一刻发生了什么,用三个 tail 命令分层排错。下面 4 节按这个顺序展开。

### 5.1 nginx 日志轮转

&emsp;&emsp;先把"轮转"这个词拆清楚:服务器上的日志文件如果不管,会一直追加,体积慢慢变大。为了让单文件体积可控、定期清理旧文件,我们用 Linux 自带的 <b>logrotate</b> 工具——它按规则自动做两件事:文件按天或按大小切开成新文件、老文件超过保留份数压缩或删除。整个过程系统每天自动跑一遍 `/etc/logrotate.d/` 下的所有规则,不需要手动管。

> <font size=2>【名词解释】<b><font color=red>logrotate</font>(日志轮转工具)</b>:Linux 自带的日志切分归档工具。按天或按大小切分日志、压缩并保留最近 N 份,防止单文件膨胀写满磁盘。规则放在 `/etc/logrotate.d/` 下,系统每天自动扫一遍执行所有规则。</font>

&emsp;&emsp;Ubuntu apt 装 nginx 时已经把 nginx 日志的轮转规则自动放在 `/etc/logrotate.d/nginx`,我们不用配,看一眼确认它在就行:

```bash
cat /etc/logrotate.d/nginx
```

&emsp;&emsp;输出里能看到 `daily` / `rotate 14` / `compress` 这几条——每天切一次、保留 14 份历史、老的压缩。nginx 这层有了,接下来给 ai-todo 也加一份。

### 5.2 应用日志补轮转

&emsp;&emsp;ai-todo 跑起来后日志写到 `/home/ubuntu/ai-todo/logs/request.log`(每次请求一行)。nginx 那份规则不管这个路径,我们给它单写一份——规则语义跟 nginx 那份一样(daily / 保留 14 份 / 压缩),路径换成 ai-todo 自己的 `logs/` 目录。

```bash
# 服务器:写一份 ai-todo 日志的 logrotate 规则
sudo tee /etc/logrotate.d/ai-todo > /dev/null <<'EOF'
/home/ubuntu/ai-todo/logs/*.log {
    su ubuntu ubuntu
    daily
    rotate 14
    compress
    missingok
    copytruncate
}
EOF
```

&emsp;&emsp;heredoc 复制到 `EOF` 就停手,下面是另一条独立的命令——强制立刻轮转一次验证配置生效:

```bash
sudo logrotate -f /etc/logrotate.d/ai-todo
ls -lah /home/ubuntu/ai-todo/logs/    # 期望多一份 request.log.1
```

&emsp;&emsp;配置里只有一个细节要单说——<b>`su ubuntu ubuntu`</b>:因为 `/home/ubuntu/ai-todo/logs/` owner 不是 root,logrotate 默认会拒绝处理这种目录,这一行明确告诉它"以 ubuntu 身份做轮转"就好。不加会报 `parent directory has insecure permissions ... Set "su" directive in config file`。

### 5.3 日志三板斧

&emsp;&emsp;部署完之后日志分三层，对应三个 tail 命令——出问题时我们会顺序排查这三层。

```bash
# 板斧 1：systemd 接管的 ai-todo 进程日志（stdout/stderr 都进 journal）
sudo journalctl -fu ai-todo

# 板斧 2：nginx 访问日志（看请求是否到了 nginx、状态码是多少）
sudo tail -f /var/log/nginx/myagent-access.log

# 板斧 3：ai-todo 应用层日志（看业务逻辑、SQL、异常栈）
sudo tail -f /var/log/ai-todo/*.log
```

&emsp;&emsp;排错时的常见思路：① 板斧 2 看 nginx 有没有收到这个请求、回了什么状态码——如果 nginx 没收到，问题在 DNS、客户端、防火墙这一段；② 板斧 1 看 ai-todo 进程是不是活着、有没有崩——systemd journal 是排查应用启动失败的第一现场；③ 板斧 3 看业务逻辑层的错误——SQL 失败、LLM 调用失败、参数校验失败都在这里。三层逐层排是工业标准节奏。

### 5.4 监控入口设计

&emsp;&emsp;最后一件事——给外部监控工具接入 `/health` 探针。ai-todo 的 `/health` 端点是部署层探针的标准入口：正常返 200 + `{"status": "ok"}`，有问题(env 缺失 / 数据库连不上)返 503 + `{"status": "degraded", "issues": [...]}` 带问题描述。这是个"轻探针"——只查本地 SQLite 和 env，不调 LLM 也不依赖外网，每分钟跑一次完全没负担。本门课不展开自托管监控面板的搭建（那是另一门课的事），但我们让 /health 这条路通到外部，可以接入任何 HTTPS 健康检查服务。

```bash
# macOS / Linux / Git Bash：验证 /health 探针外部可达
curl -fsS https://myagent-lab.online/health | python3 -m json.tool
# 期望：{"status": "ok"} 或 {"status": "degraded", "issues": [...]}
```

```bash
# Windows PowerShell：用 curl.exe + python（不是 python3）
curl.exe -fsS https://myagent-lab.online/health | python -m json.tool
```

&emsp;&emsp;`-fsS` 三个 flag 的组合：`-f` 让 HTTP 非 2xx 状态码返回非零退出码（适合脚本检测）；`-s` 静默不打印进度；`-S` 出错时仍打印 error message。这是脚本化健康检查的标准 curl 用法。`python3 -m json.tool` 把响应美化打印——把它接到任何外部监控的 HTTP 探针配置上（探针每 60 秒打一次这个 URL，看返回 200 + status=ok 就算健康），这条 ai-todo 的健康检查链路就全通了。

> <font size=2>【名词解释】<b><font color=red>uptime-kuma</font>(开源监控告警面板)</b>:开源的自托管站点监控面板,装在另一台机器上定时探活 `/health` 等端点,失败时邮件或 Webhook 告警。跟 UptimeRobot 等 SaaS 同类。</font>

&emsp;&emsp;日志能看、健康检查能探，基础观测有了。**从"上线就完"到"日志切着 + /health 探着 + 监控外接 = 7×24 可观测"——这一章我们补齐了让服务长期活下去的最后一块基础。** 我们一路从 Ubuntu 基线、代码上传、systemd 守护、HTTPS 推导、nginx 入口、nginx 扩展走到日志监控，完成了 ai-todo 从本机到 `https://myagent-lab.online` 的完整搬迁。下一章我们把这一路走过的产物、关键提示、下一步建议做一次完整回顾。

---

## <center>第六章 课程回顾</center>

&emsp;&emsp;走到这里我们已经把 ai-todo 从本机搬到了 `https://myagent-lab.online`,HTTPS 已上、SSE 流式响应实时、systemd 守着、日志切着——一台合格的公网服务。这一章我们把整条搬迁路径回放一遍,看一看走过了哪些里程碑、留下了哪些可复用的产物、未来要往哪走。

&emsp;&emsp;整条路径上的 6 个里程碑按时间线排开:基线装包 → 代码上传 → uvicorn HTTP 裸跑 → systemd 守护 → nginx 入口加 HTTPS → 日志监控,终点是浏览器以 HTTPS 打开 ai-todo + SSE 一段段实时推送的最终形态。这条时间线既是回顾,也是下次给新机器起服务时的最短路径——每一格都有一个产物,产物都能复用。

<div align=center><font size=2 color=#999999>6 个里程碑的时间线终点是 HTTPS + SSE 流式响应</font></div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/agent-deploy/2026-05-15/L7-recap-f20eff50.png" width=80% alt="6 个里程碑的时间线终点是 HTTPS + SSE 流式响应"></div>

<br>

&emsp;&emsp;我们一路从 Ubuntu 基线、代码上传、systemd 守护、HTTPS 推导、nginx 入口、nginx 扩展走到日志监控，完成了 ai-todo 从本机到 `https://myagent-lab.online` 的完整搬迁。本门课的核心叙事是"反向代理优先"：让 HTTP 先通看见不安全，再推导出 HTTPS 与入口需求，最后 nginx 作为入口承担 TLS 卸载 + SSE 透传 + 限流三件事。这一章我们做五件回顾：6 件产物 self-check、完整命令清单、整体架构图、4 个提示速查、下一步往哪走。

### 6.1 6 件产物自检清单

&emsp;&emsp;对照学习契约里的 6 件产物，逐条验证自己手上的真实状态。每一条都有一个可立刻跑的命令。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>6 件产物 self-check 清单</font></p>
<div class="center">

| # | 产物 | 自检命令 |
|---|---|---|
| ① | HTTPS 已上的 ai-todo 服务 | `curl -I https://myagent-lab.online/health` 看 200 + 浏览器显示安全连接 |
| ② | SSH 密码登录入口 + 公钥免密 | `ssh myagent` 能免密直接进 |
| ③ | systemd 守护进程 | `systemctl status ai-todo` 看 active + `kill -9` 验证自动重启 |
| ④ | nginx 完整 site config | `sudo nginx -t` 通过 + 含 HTTPS + SSE + 限流 |
| ⑤ | certbot 自动续期机制 | `systemctl list-timers \| grep certbot` 看 timer 已排 + `certbot renew --dry-run` 通过 |
| ⑥ | logrotate + /health 探针 | `logrotate -d /etc/logrotate.d/ai-todo` 通过 + `curl https://.../health` 返回 JSON |

</div>

&emsp;&emsp;6 条全部通过就是完整产物达成。任何一条没过都对应到具体章节回去查——譬如 ③ 不过就去第三章写完整 unit 文件那节重看 unit 内容;④ 不过就去第四章 nginx 限流叠加 + 防爬那节核对最终 site config。这张表不仅是验收清单,也是排错地图。

### 6.2 完整命令清单

&emsp;&emsp;把全课关键命令收成一个清单，下次新部署一台机器可以直接照着跑。这是上线后定期跑一遍的体检命令 + 新机器复刻部署的最短路径，按章节分组——前四段（一、二、三、四）是新机器搬迁路径，后四段（五、六、七、八）是 nginx/HTTPS/日志/验证。建议复制到本机 `deploy-checklist.md` 留底；其中 systemd unit 文件和 nginx site config 不在清单里，需要回对应章节填完整版再执行。

```bash
# ============================================
# 一、基线装包（在新服务器上一次性跑）
# ============================================
sudo apt update
sudo apt upgrade -y
sudo apt install -y python3.12 python3.12-venv git curl snapd
python3.12 --version

# ============================================
# 二、本机（macOS / Linux / Git Bash）：SSH alias + 代码打包传输
# ============================================
# 本机 ~/.ssh/config 加 Host alias 略

cd ~/projects/ai-todo
tar --exclude='.git' --exclude='.venv' --exclude='__pycache__' \
    --exclude='*.pyc' --exclude='.env' \
    -czf /tmp/ai-todo.tar.gz .
scp /tmp/ai-todo.tar.gz myagent:/tmp/
scp .env myagent:/tmp/ai-todo.env
# Windows PowerShell 等价:tar/scp 改 tar.exe/scp.exe,本机路径换成 D:\project\ai-todo,续行 \ 改反引号 `

# ============================================
# 三、服务器:venv + 装依赖 + 安置 .env
# ============================================
mkdir -p /home/ubuntu/ai-todo
cd /home/ubuntu/ai-todo
tar -xzf /tmp/ai-todo.tar.gz
python3.12 -m venv .venv
.venv/bin/pip install --upgrade pip
.venv/bin/pip install -r requirements.txt
.venv/bin/pip install gunicorn uvicorn-worker      # 生产运行时:gunicorn + uvicorn worker
mv /tmp/ai-todo.env /home/ubuntu/ai-todo/.env
chmod 600 /home/ubuntu/ai-todo/.env                # 只属主可读写,同机器其他账号读不到 OPENROUTER_API_KEY

# ============================================
# 四、systemd unit 写入 + 启用(ExecStart 跑 gunicorn -k uvicorn_worker.UvicornWorker)
# ============================================
# /etc/systemd/system/ai-todo.service 见 systemd 守护那一章完整 unit(用 sudo tee heredoc 写入)
sudo systemctl daemon-reload
sudo systemctl enable --now ai-todo
sudo systemctl status ai-todo --no-pager

# ============================================
# 五、nginx 装 + 启用 site + 后端收回 loopback
# ============================================
sudo apt install -y nginx
sudo rm -f /etc/nginx/sites-enabled/default
# 基础生产 site config 见 nginx 那一章 sudo tee(前端 alias + SSE 两端点透传 + 业务正则)
# 限流和防爬叠加见下一节 sudo tee(zone 定义 + 完整生产 site config 覆盖)
sudo ln -sf /etc/nginx/sites-available/myagent-lab.online \
            /etc/nginx/sites-enabled/
# 开放 nginx 访问前端目录的权限(默认 /home/ubuntu 是 750,www-data 进不去)
sudo chmod o+x /home/ubuntu
sudo chmod -R o+rX /home/ubuntu/ai-todo/frontend
# 后端 bind 从 0.0.0.0:12234 改成 127.0.0.1:12234(收回 loopback,公网不直连后端)
sudo sed -i 's|--bind 0.0.0.0:12234|--bind 127.0.0.1:12234|' \
    /etc/systemd/system/ai-todo.service
sudo systemctl daemon-reload
sudo systemctl restart ai-todo
sudo pkill -f "http.server 12233"                  # kill 临时前端 web server
sudo nginx -t
sudo systemctl reload nginx

# ============================================
# 六、certbot 签证 + 自动续期
# ============================================
sudo snap install --classic certbot
sudo ln -sf /snap/bin/certbot /usr/bin/certbot
sudo apt remove -y certbot python3-certbot-nginx || true
sudo certbot --nginx -d myagent-lab.online --redirect \
    --agree-tos --register-unsafely-without-email
sudo certbot renew --dry-run

# ============================================
# 七、logrotate 配应用层日志轮转
# ============================================
# 把日志那一章的 sudo tee /etc/logrotate.d/ai-todo heredoc 跑一遍后,强制轮转一次验证
sudo logrotate -f /etc/logrotate.d/ai-todo

# ============================================
# 八、健康检查端到端验证
# ============================================
curl -i https://myagent-lab.online/health          # 注意小写 -i,大写 -I 是 HEAD,FastAPI @app.get 不响应 HEAD
curl -fsS https://myagent-lab.online/health | python3 -m json.tool
```

&emsp;&emsp;这八段命令是从空白服务器到完整部署的最短路径。建议把它复制到本机 `deploy-checklist.md` 留底，下次起新服务换个 `server_name` 和域名就能复用——所有组件都通用，只有少数几个名字要替换。

### 6.3 整体架构图

&emsp;&emsp;开课前我们脑子里设想的部署拓扑是这样的——浏览器经过域名解析打到一台公网服务器,服务器里 nginx 把请求往后端 gunicorn(prefork uvicorn worker)转,业务接口下面接 LangGraph 跑模型。一句话就这一条主线,但每个节点该装什么、用什么版本、怎么配,当时都是空的。

<div align=center><font size=2 color=#999999>开课前脑子里的部署拓扑——主线清晰但节点都还是空的</font></div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/agent-deploy/2026-05-15/L1-topology-ac0a4c62.png" width=80% alt="开课前脑子里的部署拓扑——主线清晰但节点都还是空的"></div>

<br>

&emsp;&emsp;走完六章之后,这张骨架还是一样,但每个节点都从空盒子填满了具体的配置和版本号:nginx 1.24 + Let's Encrypt 90 天自动续 + 三组 location(精确匹配 SSE 端点 + 正则匹配业务路由 + 普通前缀兜底静态) + SSE 三件套防御性配置 + 入口限流 + 防爬;gunicorn + uvicorn worker 跑在 `127.0.0.1:12234` loopback、systemd 守护;logrotate 切日志;`/health` 暴露给外部监控。请求链路完整跑一遍是:浏览器请求 `https://myagent-lab.online` → DNS 解析到 VPS 公网 IP → nginx 在 443 端口做 TLS 卸载 → 按 location 把请求分流(`/chat/events` / `/chat/stream` 走精确匹配 + SSE 透传、`/sessions` / `/todos` / `/health` 走正则反代、`/` 走静态兜底到 `frontend/`)→ gunicorn 在 `127.0.0.1:12234` 接请求 → 业务接口下面 LangGraph agent 调 OpenRouter。所有日志归三处(journalctl / nginx access log / ai-todo 应用 log),logrotate 统一切分。`/health` 是外部监控的接入点。

### 6.4 下一步往哪走

&emsp;&emsp;本门课走的是从本机到 https://域名 的**最小最稳**路径——单机、单一应用、HTTPS 卸载在 nginx、自动续期、基础监控。这套是工业部署的基线，不是天花板。再深一层的话题——容器编排、自动化发布流水线、流量与监控的纵深、多节点负载——是另一门课的事。先把这套基础搬迁路径吃透，后续走哪都有底。

&emsp;&emsp;回到课程一开始那张拓扑图——我们今天走过的每一步都在它上面留下了印记。`https://myagent-lab.online` 这个地址不只是一个域名，它是一整套基础设施串起来的产物。把这些产物保管好——一份 `nginx.conf`、一份 `ai-todo.service`、一份 `.env`、一份 `logrotate.d/ai-todo`——下次要起一个新服务，这套模板换个 `server_name` 就能复用。